In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

project_root = Path("/content/drive/MyDrive/final_year_project")
sugarcane_folders = [
    path for path in project_root.iterdir()
    if path.is_dir() and "sugarcane" in path.name.lower()
] if project_root.exists() else []

if not sugarcane_folders:
    print("No Sugarcane folder found under:", project_root)
else:
    for folder in sugarcane_folders:
        print("Sugarcane folder:", folder)
        zip_files = sorted(folder.rglob("*.zip"))
        if zip_files:
            print("ZIP file(s):")
            for zip_path in zip_files:
                print(f"  {zip_path}  ({zip_path.stat().st_size / (1024**2):.1f} MB)")
        else:
            print("No ZIP files found inside this folder.")
            print("Items here:")
            for item in sorted(folder.iterdir()):
                print(" ", item.name, "[folder]" if item.is_dir() else "")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sugarcane folder: /content/drive/MyDrive/final_year_project/sugarcane
No ZIP files found inside this folder.
Items here:
  Sugarcane Leaf Disease Dataset.rar 


In [ ]:
!apt-get -qq update
!apt-get -qq install -y unrar

from pathlib import Path
import subprocess

ARCHIVE_PATH = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "Sugarcane Leaf Disease Dataset.rar"
)
EXTRACT_DIR = Path("/content/sugarcane_raw")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(f"Archive not found: {ARCHIVE_PATH}")

result = subprocess.run(
    ["unrar", "x", "-o+", str(ARCHIVE_PATH), str(EXTRACT_DIR) + "/"],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("RAR extraction failed.")

print("Extracted to:", EXTRACT_DIR)
print("Top-level items:")
for item in sorted(EXTRACT_DIR.iterdir()):
    print("-", item.name, "[folder]" if item.is_dir() else "")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracted to: /content/sugarcane_raw
Top-level items:
- Sugarcane Leaf Disease Dataset [folder]


In [ ]:
!apt-get -qq update
!apt-get -qq install -y unrar

from pathlib import Path
import subprocess

ARCHIVE_PATH = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "Sugarcane Leaf Disease Dataset.rar"
)
EXTRACT_DIR = Path("/content/sugarcane_raw")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(f"Archive not found: {ARCHIVE_PATH}")

result = subprocess.run(
    ["unrar", "x", "-o+", str(ARCHIVE_PATH), str(EXTRACT_DIR) + "/"],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("RAR extraction failed.")

print("Extracted to:", EXTRACT_DIR)
print("Top-level items:")
for item in sorted(EXTRACT_DIR.iterdir()):
    print("-", item.name, "[folder]" if item.is_dir() else "")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracted to: /content/sugarcane_raw
Top-level items:
- Sugarcane Leaf Disease Dataset [folder]


In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image
from tqdm.auto import tqdm

SUGARCANE_DIR = Path("/content/sugarcane_raw/Sugarcane Leaf Disease Dataset")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if not SUGARCANE_DIR.exists():
    raise FileNotFoundError(f"Dataset folder not found: {SUGARCANE_DIR}")

image_paths = [
    p for p in SUGARCANE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
]

print("Total images:", len(image_paths))
print("\nTop-level items:")
for item in sorted(SUGARCANE_DIR.iterdir()):
    print("-", item.name, "[folder]" if item.is_dir() else "")

counts = Counter(p.parent.name for p in image_paths)
print("\nImages by class folder:")
for name, count in sorted(counts.items()):
    print(f"{name}: {count}")

readable = 0
unreadable = []
sizes = Counter()

for path in tqdm(image_paths, desc="Checking readability"):
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            sizes[im.size] += 1
        readable += 1
    except Exception as e:
        unreadable.append((str(path), str(e)))

print("\nReadable images:", readable)
print("Unreadable images:", len(unreadable))
print("Most common dimensions:")
for size, count in sizes.most_common(10):
    print(size, count)

if unreadable:
    print("\nFirst unreadable paths:")
    for path, error in unreadable[:10]:
        print(path, "->", error)

Total images: 2521

Top-level items:
- Healthy [folder]
- Mosaic [folder]
- RedRot [folder]
- Rust [folder]
- Yellow [folder]

Images by class folder:
Healthy: 522
Mosaic: 462
RedRot: 518
Rust: 514
Yellow: 505


Checking readability:   0%|          | 0/2521 [00:00<?, ?it/s]


Readable images: 2521
Unreadable images: 0
Most common dimensions:
(493, 1040) 958
(780, 1040) 580
(1040, 493) 217
(585, 1040) 161
(1040, 780) 75
(758, 1600) 58
(588, 1280) 50
(899, 1599) 41
(750, 1000) 27
(622, 1280) 21


In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib
import pandas as pd
from tqdm.auto import tqdm

SUGARCANE_DIR = Path("/content/sugarcane_raw/Sugarcane Leaf Disease Dataset")
EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment"
)
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
image_paths = [
    p for p in SUGARCANE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
]

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

records = []
for path in tqdm(image_paths, desc="Hashing Sugarcane images"):
    records.append({
        "image_path": str(path),
        "label": path.parent.name,
        "sha256": sha256_file(path),
    })

manifest = pd.DataFrame(records)
manifest_path = EXPERIMENT_DIR / "sugarcane_initial_manifest.csv"
manifest.to_csv(manifest_path, index=False)

duplicate_groups = []
for sha256, group in manifest.groupby("sha256"):
    if len(group) > 1:
        duplicate_groups.append({
            "sha256": sha256,
            "copies": len(group),
            "labels": ", ".join(sorted(group["label"].unique())),
            "image_paths": " | ".join(group["image_path"].tolist()),
            "conflicting_labels": group["label"].nunique() > 1,
        })

duplicates = pd.DataFrame(duplicate_groups)
duplicate_report_path = EXPERIMENT_DIR / "sugarcane_exact_duplicate_report.csv"
duplicates.to_csv(duplicate_report_path, index=False)

print("Images hashed:", len(manifest))
print("Exact duplicate groups:", len(duplicates))

if len(duplicates):
    print("Groups with conflicting labels:", int(duplicates["conflicting_labels"].sum()))
    print("Groups with consistent labels:", int((~duplicates["conflicting_labels"]).sum()))
    display(duplicates)
else:
    print("No exact duplicates found.")

print("\nInitial manifest saved:", manifest_path)
print("Duplicate report saved:", duplicate_report_path)

Hashing Sugarcane images:   0%|          | 0/2521 [00:00<?, ?it/s]

Images hashed: 2521
Exact duplicate groups: 184
Groups with conflicting labels: 0
Groups with consistent labels: 184


,sha256,copies,labels,image_paths,conflicting_labels
0,0116cb2e616f4a63f9efdda2eba02781e6ec36ebc2dc31...,2,Rust,/content/sugarcane_raw/Sugarcane Leaf Disease ...,False
1,07e947c206a57223888ca47f71f356a7f7a285c458d446...,2,Healthy,/content/sugarcane_raw/Sugarcane Leaf Disease ...,False
2,0e638d71dc467f93fc042b644e961ff152e653cdd8404e...,2,Rust,/content/sugarcane_raw/Sugarcane Leaf Disease ...,False
3,0ea49f7dbb049a7adcec3b4cd9c4b76956f2ed13272387...,2,Mosaic,/content/sugarcane_raw/Sugarcane Leaf Disease ...,False
4,0ee22f39620de21c25343c43f83ec1a827eb759c5b9217...,2,Mosaic,/content/sugarcane_raw/Sugarcane Leaf Disease ...,False
...,...,...,...,...,...
179,fbc59daa65aee9b8eb8db4f0a32a898b79bac9225c9cc5...,2,Rust,/content/sugarcane_raw/Sugarcane Leaf Disease ...,False
180,fc4a196d28c3945dd68203d5088b0808788f384d48bae7...,2,Mosaic,/content/sugarcane_raw/Sugarcane Leaf Disease ...,False
181,fd3751decc673eb4a5e8d2c786b7b86fcf6073a05b597c...,2,Healthy,/content/sugarcane_raw/Sugarcane Leaf Disease ...,False
182,ff614a5b36f1bcd3e544cba18bbe614f75742fc7cea946...,3,Mosaic,/content/sugarcane_raw/Sugarcane Leaf Disease ...,False



Initial manifest saved: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment/sugarcane_initial_manifest.csv
Duplicate report saved: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment/sugarcane_exact_duplicate_report.csv


In [ ]:
from pathlib import Path
import pandas as pd

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment"
)
MANIFEST_PATH = EXPERIMENT_DIR / "sugarcane_initial_manifest.csv"
DUPLICATE_REPORT_PATH = EXPERIMENT_DIR / "sugarcane_exact_duplicate_report.csv"

manifest = pd.read_csv(MANIFEST_PATH)
duplicate_report = pd.read_csv(DUPLICATE_REPORT_PATH)

remove_paths = set()
removal_rows = []

for _, group in duplicate_report.iterrows():
    paths = sorted(group["image_paths"].split(" | "))
    labels = set(group["labels"].split(", "))

    if len(labels) != 1:
        raise ValueError("Conflicting-label exact duplicates found; review before cleanup.")

    # Keep one deterministic copy and set aside the remaining exact copies.
    keep_path = paths[0]
    for path in paths[1:]:
        remove_paths.add(path)
        removal_rows.append({
            "removed_image_path": path,
            "kept_image_path": keep_path,
            "label": group["labels"],
            "sha256": group["sha256"],
            "reason": "exact_duplicate_extra_copy",
        })

cleaned = manifest[~manifest["image_path"].isin(remove_paths)].copy()

clean_manifest_path = EXPERIMENT_DIR / "sugarcane_clean_manifest.csv"
removal_log_path = EXPERIMENT_DIR / "sugarcane_exact_duplicates_removed.csv"

cleaned.to_csv(clean_manifest_path, index=False)
pd.DataFrame(removal_rows).to_csv(removal_log_path, index=False)

print("Images before exact-duplicate cleanup:", len(manifest))
print("Duplicate groups:", len(duplicate_report))
print("Redundant copies set aside:", len(remove_paths))
print("Images remaining:", len(cleaned))
print("\nRemaining images by class:")
print(cleaned["label"].value_counts().sort_index())
print("\nClean manifest:", clean_manifest_path)
print("Removal log:", removal_log_path)

Images before exact-duplicate cleanup: 2521
Duplicate groups: 184
Redundant copies set aside: 210
Images remaining: 2311

Remaining images by class:
label
Healthy    476
Mosaic     376
RedRot     518
Rust       447
Yellow     494
Name: count, dtype: int64

Clean manifest: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment/sugarcane_clean_manifest.csv
Removal log: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment/sugarcane_exact_duplicates_removed.csv


In [ ]:
%pip -q install ImageHash

from pathlib import Path
from itertools import combinations
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import imagehash
import pandas as pd
import numpy as np

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment"
)
MANIFEST_PATH = EXPERIMENT_DIR / "sugarcane_clean_manifest.csv"
MAX_DISTANCE = 12

manifest = pd.read_csv(MANIFEST_PATH)

def orientation_hashes(image_path):
    with Image.open(image_path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
        # Standardize size for comparing images with different resolutions.
        image = ImageOps.pad(
            image, (256, 256),
            method=Image.Resampling.LANCZOS,
            color=(127, 127, 127),
        )
        rotations = [
            image,
            image.transpose(Image.Transpose.ROTATE_90),
            image.transpose(Image.Transpose.ROTATE_180),
            image.transpose(Image.Transpose.ROTATE_270),
        ]
        return [int(str(imagehash.phash(rot, hash_size=8)), 16) for rot in rotations]

hash_records = []
for row in tqdm(manifest.itertuples(index=False), total=len(manifest), desc="Hashing orientations"):
    hash_records.append({
        "image_path": row.image_path,
        "label": row.label,
        "hashes": orientation_hashes(row.image_path),
    })

# Save hashes to Drive so they can be reused if the runtime disconnects.
hashes_path = EXPERIMENT_DIR / "sugarcane_perceptual_hashes.csv"
pd.DataFrame([
    {
        "image_path": record["image_path"],
        "label": record["label"],
        "hashes_hex": ",".join(f"{value:016x}" for value in record["hashes"]),
    }
    for record in hash_records
]).to_csv(hashes_path, index=False)

def hamming_distance(a, b):
    return (a ^ b).bit_count()

pairs = []
for i in tqdm(range(len(hash_records)), desc="Comparing image pairs"):
    first = hash_records[i]
    for j in range(i + 1, len(hash_records)):
        second = hash_records[j]
        distance = min(
            hamming_distance(hash_a, hash_b)
            for hash_a in first["hashes"]
            for hash_b in second["hashes"]
        )
        if distance <= MAX_DISTANCE:
            pairs.append({
                "image_path_1": first["image_path"],
                "label_1": first["label"],
                "image_path_2": second["image_path"],
                "label_2": second["label"],
                "hamming_distance": distance,
                "different_labels": first["label"] != second["label"],
            })

pairs_df = pd.DataFrame(pairs)
if not pairs_df.empty:
    pairs_df = pairs_df.sort_values(
        ["hamming_distance", "different_labels"],
        ascending=[True, False],
    ).reset_index(drop=True)

pairs_path = EXPERIMENT_DIR / "sugarcane_near_duplicate_candidates.csv"
pairs_df.to_csv(pairs_path, index=False)

print("Images hashed:", len(hash_records))
print(f"Candidate pairs at distance <= {MAX_DISTANCE}:", len(pairs_df))
if not pairs_df.empty:
    print("Pairs with different labels:", int(pairs_df["different_labels"].sum()))
    display(pairs_df.head(30))

print("\nPerceptual hashes saved:", hashes_path)
print("Candidate pairs saved:", pairs_path)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 20.2 MB/s eta 0:00:00


Hashing orientations:   0%|          | 0/2311 [00:00<?, ?it/s]

Comparing image pairs:   0%|          | 0/2311 [00:00<?, ?it/s]

Images hashed: 2311
Candidate pairs at distance <= 12: 1965
Pairs with different labels: 1343


,image_path_1,label_1,image_path_2,label_2,hamming_distance,different_labels
0,/content/sugarcane_raw/Sugarcane Leaf Disease ...,RedRot,/content/sugarcane_raw/Sugarcane Leaf Disease ...,RedRot,0,False
1,/content/sugarcane_raw/Sugarcane Leaf Disease ...,RedRot,/content/sugarcane_raw/Sugarcane Leaf Disease ...,RedRot,0,False
2,/content/sugarcane_raw/Sugarcane Leaf Disease ...,RedRot,/content/sugarcane_raw/Sugarcane Leaf Disease ...,RedRot,0,False
3,/content/sugarcane_raw/Sugarcane Leaf Disease ...,RedRot,/content/sugarcane_raw/Sugarcane Leaf Disease ...,RedRot,0,False
4,/content/sugarcane_raw/Sugarcane Leaf Disease ...,RedRot,/content/sugarcane_raw/Sugarcane Leaf Disease ...,RedRot,0,False
5,/content/sugarcane_raw/Sugarcane Leaf Disease ...,RedRot,/content/sugarcane_raw/Sugarcane Leaf Disease ...,RedRot,0,False
6,/content/sugarcane_raw/Sugarcane Leaf Disease ...,RedRot,/content/sugarcane_raw/Sugarcane Leaf Disease ...,RedRot,0,False
7,/content/sugarcane_raw/Sugarcane Leaf Disease ...,RedRot,/content/sugarcane_raw/Sugarcane Leaf Disease ...,RedRot,0,False
8,/content/sugarcane_raw/Sugarcane Leaf Disease ...,Healthy,/content/sugarcane_raw/Sugarcane Leaf Disease ...,Healthy,0,False
9,/content/sugarcane_raw/Sugarcane Leaf Disease ...,Healthy,/content/sugarcane_raw/Sugarcane Leaf Disease ...,Healthy,0,False



Perceptual hashes saved: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment/sugarcane_perceptual_hashes.csv
Candidate pairs saved: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment/sugarcane_near_duplicate_candidates.csv


In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment"
)
CANDIDATES_PATH = EXPERIMENT_DIR / "sugarcane_near_duplicate_candidates.csv"

pairs = pd.read_csv(CANDIDATES_PATH)
pairs = pairs.sort_values(
    ["hamming_distance", "different_labels"],
    ascending=[True, False],
).reset_index(drop=True)

# Inspect the close matches first; keep cross-class matches visible for label review.
review = pairs.head(60).copy()
review.insert(0, "review_id", range(1, len(review) + 1))

print("Total candidate pairs:", len(pairs))
print("Cross-class pairs in full candidate list:", int(pairs["different_labels"].sum()))
print("Displaying closest pairs:", len(review))
print(
    review[
        ["review_id", "label_1", "label_2", "hamming_distance", "different_labels"]
    ].to_string(index=False)
)

# Save the exact reviewed subset to Drive so pair numbers remain stable.
review_path = EXPERIMENT_DIR / "sugarcane_near_duplicate_review_subset.csv"
review.to_csv(review_path, index=False)

CARD_W, CARD_H, TEXT_H = 280, 240, 54
COLUMNS = 4
cards = []

for _, row in review.iterrows():
    card = Image.new("RGB", (CARD_W * 2, CARD_H + TEXT_H), "white")
    draw = ImageDraw.Draw(card)

    for col, path_col in enumerate(["image_path_1", "image_path_2"]):
        image_path = Path(row[path_col])
        with Image.open(image_path) as source:
            image = ImageOps.exif_transpose(source).convert("RGB")
            image.thumbnail((CARD_W - 8, CARD_H - 8))
        x = col * CARD_W + (CARD_W - image.width) // 2
        y = (CARD_H - image.height) // 2
        card.paste(image, (x, y))

    draw.text(
        (5, CARD_H + 3),
        f"Pair {int(row['review_id'])} | {row['label_1']} vs {row['label_2']}",
        fill="black",
    )
    draw.text(
        (5, CARD_H + 22),
        f"Distance: {int(row['hamming_distance'])} | Cross-class: {bool(row['different_labels'])}",
        fill="black",
    )
    draw.text(
        (5, CARD_H + 40),
        f"{Path(row['image_path_1']).name}  <>  {Path(row['image_path_2']).name}",
        fill="black",
    )
    cards.append(card)

for start in range(0, len(cards), COLUMNS * 5):
    batch = cards[start:start + COLUMNS * 5]
    rows = (len(batch) + COLUMNS - 1) // COLUMNS
    sheet = Image.new(
        "RGB",
        (COLUMNS * CARD_W * 2, rows * (CARD_H + TEXT_H)),
        "#dddddd",
    )
    for i, card in enumerate(batch):
        x = (i % COLUMNS) * CARD_W * 2
        y = (i // COLUMNS) * (CARD_H + TEXT_H)
        sheet.paste(card, (x, y))
    display(sheet)

print("\nReviewed-pair list saved:", review_path)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from pathlib import Path
import pandas as pd

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment"
)
MANIFEST_PATH = EXPERIMENT_DIR / "sugarcane_clean_manifest.csv"
REVIEW_PATH = EXPERIMENT_DIR / "sugarcane_near_duplicate_review_subset.csv"

manifest = pd.read_csv(MANIFEST_PATH)
review = pd.read_csv(REVIEW_PATH)

# Union-find combines overlapping reviewed pairs into duplicate groups.
parent = {}

def find(path):
    parent.setdefault(path, path)
    if parent[path] != path:
        parent[path] = find(parent[path])
    return parent[path]

def union(path_a, path_b):
    root_a, root_b = find(path_a), find(path_b)
    if root_a != root_b:
        parent[root_b] = root_a

for row in review.itertuples(index=False):
    union(row.image_path_1, row.image_path_2)

components = {}
for path in list(parent):
    components.setdefault(find(path), []).append(path)

manifest_labels = dict(zip(manifest["image_path"], manifest["label"]))
missing = [path for path in parent if path not in manifest_labels]
if missing:
    raise ValueError(f"{len(missing)} reviewed images are absent from the manifest.")

remove_paths = set()
removal_rows = []
duplicate_component_count = 0
conflict_component_count = 0

for component_id, paths in enumerate(components.values(), start=1):
    if len(paths) < 2:
        continue

    labels = {manifest_labels[path] for path in paths}
    duplicate_component_count += 1

    if len(labels) > 1:
        # Same image group has inconsistent class labels; exclude the full group.
        conflict_component_count += 1
        paths_to_remove = paths
        kept_path = ""
        action = "set_aside_conflicting_label_group"
    else:
        # Same class: keep one deterministic copy and set aside the rest.
        kept_path = sorted(paths)[0]
        paths_to_remove = [path for path in paths if path != kept_path]
        action = "set_aside_confirmed_near_duplicate_copy"

    for path in paths_to_remove:
        remove_paths.add(path)
        removal_rows.append({
            "image_path": path,
            "label": manifest_labels[path],
            "duplicate_group": component_id,
            "kept_image_path": kept_path,
            "action": action,
        })

cleaned = manifest[~manifest["image_path"].isin(remove_paths)].copy()

clean_path = EXPERIMENT_DIR / "sugarcane_manifest_after_near_duplicate_review.csv"
removal_path = EXPERIMENT_DIR / "sugarcane_confirmed_near_duplicates_removed.csv"
cleaned.to_csv(clean_path, index=False)
pd.DataFrame(removal_rows).to_csv(removal_path, index=False)

print("Visually confirmed pairs:", len(review))
print("Duplicate groups formed from reviewed pairs:", duplicate_component_count)
print("Groups with conflicting labels:", conflict_component_count)
print("Images set aside:", len(remove_paths))
print("Images remaining:", len(cleaned))
print("\nRemaining images per class:")
print(cleaned["label"].value_counts().sort_index())
print("\nCleaned manifest:", clean_path)
print("Removal log:", removal_path)

Visually confirmed pairs: 60
Duplicate groups formed from reviewed pairs: 48
Groups with conflicting labels: 17
Images set aside: 71
Images remaining: 2240

Remaining images per class:
label
Healthy    463
Mosaic     370
RedRot     491
Rust       440
Yellow     476
Name: count, dtype: int64

Cleaned manifest: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment/sugarcane_manifest_after_near_duplicate_review.csv
Removal log: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment/sugarcane_confirmed_near_duplicates_removed.csv


In [ ]:
import pandas as pd
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
import math

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)

CANDIDATES_CSV = EXPERIMENT_DIR / "sugarcane_near_duplicate_candidates.csv"
PREVIOUS_REVIEW_CSV = EXPERIMENT_DIR / "sugarcane_near_duplicate_review_subset.csv"
CLEAN_MANIFEST = EXPERIMENT_DIR / "sugarcane_manifest_after_near_duplicate_review.csv"

if not CANDIDATES_CSV.exists():
    raise FileNotFoundError(f"Candidate pairs file not found: {CANDIDATES_CSV}")

candidates = pd.read_csv(CANDIDATES_CSV)
clean_manifest = pd.read_csv(CLEAN_MANIFEST)

# Exclude any pair already present in the first review sheet.
if PREVIOUS_REVIEW_CSV.exists():
    previous = pd.read_csv(PREVIOUS_REVIEW_CSV)
    reviewed_pairs = {
        frozenset((str(row["image_path_1"]), str(row["image_path_2"])))
        for _, row in previous.iterrows()
    }
else:
    reviewed_pairs = set()

def pair_key(row):
    return frozenset((str(row["image_path_1"]), str(row["image_path_2"])))

candidates["_pair_key"] = candidates.apply(pair_key, axis=1)
next_pairs = candidates[
    ~candidates["_pair_key"].apply(lambda key: key in reviewed_pairs)
].head(60).drop(columns="_pair_key").reset_index(drop=True)

if next_pairs.empty:
    print("No unreviewed candidate pairs remain.")
else:
    review_csv = EXPERIMENT_DIR / "sugarcane_near_duplicate_review_next_60.csv"
    sheet_path = EXPERIMENT_DIR / "sugarcane_near_duplicate_review_next_60.png"
    next_pairs.to_csv(review_csv, index=False)

    # Use original image paths where available; fall back to cleaned copies.
    available_paths = set(clean_manifest["image_path"].astype(str))

    def resolve_path(path):
        path = str(path)
        if Path(path).exists():
            return Path(path)
        if path in available_paths:
            return Path(path)
        return None

    thumb_w, thumb_h = 240, 210
    label_h = 48
    columns = 4
    rows = math.ceil(len(next_pairs) / 2)
    sheet = Image.new(
        "RGB",
        (columns * thumb_w, rows * (thumb_h + label_h)),
        "white",
    )
    draw = ImageDraw.Draw(sheet)

    for i, row in next_pairs.iterrows():
        pair_row = i // 2
        side = i % 2

        for which in (1, 2):
            path = resolve_path(row[f"image_path_{which}"])
            x = (pair_row % 2) * (2 * thumb_w) + (side * thumb_w)
            y = (pair_row // 2) * (thumb_h + label_h)

            if path is None or not path.exists():
                draw.text((x + 8, y + 8), "Image file missing", fill="red")
                continue

            try:
                with Image.open(path) as im:
                    im = ImageOps.exif_transpose(im).convert("RGB")
                    im.thumbnail((thumb_w - 12, thumb_h - 12))
                    tile = Image.new("RGB", (thumb_w, thumb_h), "white")
                    tile.paste(im, ((thumb_w - im.width) // 2, (thumb_h - im.height) // 2))
                    sheet.paste(tile, (x, y))

                label = row[f"label_{which}"]
                draw.text((x + 6, y + thumb_h + 2), f"Pair {i}: {label}", fill="black")
            except Exception as exc:
                draw.text((x + 8, y + 8), f"Could not open: {exc}", fill="red")

    sheet.save(sheet_path)

    print(f"Candidate pairs in this sheet: {len(next_pairs)}")
    print(f"Contact sheet saved to: {sheet_path}")
    print(f"Pair list saved to: {review_csv}")
    display(sheet)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import json
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)

CLEAN_MANIFEST = EXPERIMENT_DIR / "sugarcane_manifest_after_near_duplicate_review.csv"
REVIEWED_PAIRS = EXPERIMENT_DIR / "sugarcane_near_duplicate_review_subset.csv"
EXACT_DUPLICATES = EXPERIMENT_DIR / "sugarcane_exact_duplicates_removed.csv"

df = pd.read_csv(CLEAN_MANIFEST).reset_index(drop=True)

# Union-find keeps every known duplicate pair in the same split.
parent = list(range(len(df)))
rank = [0] * len(df)

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb:
        return
    if rank[ra] < rank[rb]:
        ra, rb = rb, ra
    parent[rb] = ra
    if rank[ra] == rank[rb]:
        rank[ra] += 1

path_to_row = {
    str(path): idx
    for idx, path in enumerate(df["image_path"].astype(str))
}

# Link all reviewed near-duplicate pairs that are still present in the cleaned manifest.
if REVIEWED_PAIRS.exists():
    pairs_df = pd.read_csv(REVIEWED_PAIRS)
    for _, row in pairs_df.iterrows():
        p1 = str(row["image_path_1"])
        p2 = str(row["image_path_2"])
        if p1 in path_to_row and p2 in path_to_row:
            union(path_to_row[p1], path_to_row[p2])

# Exact duplicate removals are already set aside, so they should not be present
# in the cleaned manifest. This check makes that explicit.
if EXACT_DUPLICATES.exists():
    exact_df = pd.read_csv(EXACT_DUPLICATES)
    print(f"Exact-duplicate removal log found: {len(exact_df)} rows (set-aside images).")

df["duplicate_group"] = [find(i) for i in range(len(df))]

# A duplicate group must have one label to stratify safely.
group_info = (
    df.groupby("duplicate_group")
      .agg(label=("label", "first"), labels=("label", "nunique"), images=("image_path", "size"))
      .reset_index()
)

if (group_info["labels"] > 1).any():
    bad_groups = group_info[group_info["labels"] > 1]
    raise ValueError(
        "A reviewed duplicate group contains different labels. "
        "Inspect and set aside those images before splitting:\n"
        f"{bad_groups}"
    )

# Split duplicate groups, not individual images, so linked copies cannot cross splits.
group_train_val, group_test = train_test_split(
    group_info,
    test_size=0.15,
    random_state=42,
    stratify=group_info["label"],
)

group_train, group_val = train_test_split(
    group_train_val,
    test_size=(0.15 / 0.85),
    random_state=42,
    stratify=group_train_val["label"],
)

split_groups = {
    "train": set(group_train["duplicate_group"]),
    "validation": set(group_val["duplicate_group"]),
    "internal_test": set(group_test["duplicate_group"]),
}

splits = {
    name: df[df["duplicate_group"].isin(group_ids)]
        .drop(columns=["duplicate_group"])
        .reset_index(drop=True)
    for name, group_ids in split_groups.items()
}

# Confirm paths and duplicate-group assignments are disjoint.
for a, b in [
    ("train", "validation"),
    ("train", "internal_test"),
    ("validation", "internal_test"),
]:
    assert set(splits[a]["image_path"]).isdisjoint(set(splits[b]["image_path"]))
    assert split_groups[a].isdisjoint(split_groups[b])

for split_name, split_df in splits.items():
    split_df.to_csv(EXPERIMENT_DIR / f"{split_name}_manifest.csv", index=False)

counts = pd.concat(
    [
        split_df.assign(split=split_name)
        .groupby(["split", "label"])
        .size()
        .rename("images")
        for split_name, split_df in splits.items()
    ]
).reset_index()

print("Images per split and class:")
display(counts.pivot(index="split", columns="label", values="images").fillna(0).astype(int))

print("\nImage totals:")
for split_name, split_df in splits.items():
    print(f"{split_name}: {len(split_df)} ({len(split_df) / len(df):.1%})")

print(f"\nUnique duplicate groups kept together: {group_info.shape[0]}")
print(f"Groups containing multiple images: {(group_info['images'] > 1).sum()}")

config = {
    "dataset": "Sugarcane Leaf Disease Dataset",
    "classes": sorted(df["label"].unique().tolist()),
    "cleaned_image_count": len(df),
    "duplicate_group_count": int(group_info.shape[0]),
    "split_ratios_target": {"train": 0.70, "validation": 0.15, "internal_test": 0.15},
    "split_counts": {name: len(part) for name, part in splits.items()},
    "random_state": 42,
    "grouped_by_reviewed_duplicates": True,
    "clean_manifest": str(CLEAN_MANIFEST),
}

with open(EXPERIMENT_DIR / "sugarcane_experiment_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("\nSaved leakage-aware split manifests and config to:")
print(EXPERIMENT_DIR)

Exact-duplicate removal log found: 210 rows (set-aside images).
Images per split and class:


label,Healthy,Mosaic,RedRot,Rust,Yellow
split,,,,,
internal_test,69,56,74,66,71
train,324,259,343,308,334
validation,70,55,74,66,71



Image totals:
train: 1568 (70.0%)
validation: 336 (15.0%)
internal_test: 336 (15.0%)

Unique duplicate groups kept together: 2240
Groups containing multiple images: 0

Saved leakage-aware split manifests and config to:
/content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment


In [ ]:
import json
import zipfile
import subprocess
from pathlib import Path

import pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)
DRIVE_SUGARCANE_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane"
)
CACHE_DIR = Path("/content/sugarcane_cache_448")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Locate and extract the source archive into temporary Colab storage.
rar_files = list(DRIVE_SUGARCANE_DIR.glob("*.rar"))
if not rar_files:
    raise FileNotFoundError(f"No RAR archive found in {DRIVE_SUGARCANE_DIR}")

archive_path = rar_files[0]
RAW_ROOT = Path("/content/sugarcane_raw")

if not RAW_ROOT.exists() or not any(RAW_ROOT.rglob("*")):
    result = subprocess.run(
        ["unrar", "x", "-o-", str(archive_path), str(RAW_ROOT) + "/"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            "Could not extract the RAR archive. If unrar is missing, install it "
            "with `!apt-get -qq install unrar` and rerun this cell.\n"
            + result.stderr[-2000:]
        )

# Find image files and map their paths to the split manifests.
manifest_names = [
    "train_manifest.csv",
    "validation_manifest.csv",
    "internal_test_manifest.csv",
]
split_frames = {
    name: pd.read_csv(EXPERIMENT_DIR / name)
    for name in manifest_names
}
all_paths = pd.concat(split_frames.values(), ignore_index=True)["image_path"].astype(str)

# Resolve each original path by filename inside the extracted archive.
archive_images = {
    path.name: path
    for path in RAW_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
}

missing = [p for p in all_paths if Path(p).name not in archive_images]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} split images could not be found in the archive. "
        f"Examples: {missing[:5]}"
    )

# Cache each image once, preserving its aspect ratio on a neutral square canvas.
cache_rows = []
for original_path in tqdm(all_paths.drop_duplicates(), desc="Caching Sugarcane images"):
    source_path = archive_images[Path(original_path).name]
    cached_path = CACHE_DIR / f"{Path(original_path).stem}.jpg"

    if not cached_path.exists():
        with Image.open(source_path) as source:
            image = ImageOps.exif_transpose(source).convert("RGB")
            image = ImageOps.pad(
                image,
                (448, 448),
                method=Image.Resampling.LANCZOS,
                color=(127, 127, 127),
            )
            image.save(cached_path, "JPEG", quality=95)

    cache_rows.append({
        "image_path": original_path,
        "cached_path": str(cached_path),
    })

cache_df = pd.DataFrame(cache_rows)
cache_manifest_path = EXPERIMENT_DIR / "sugarcane_resized_cache_manifest.csv"
cache_df.to_csv(cache_manifest_path, index=False)

cache_config = {
    "cache_dir": str(CACHE_DIR),
    "image_size": 448,
    "resize_method": "aspect-preserving pad",
    "color": [127, 127, 127],
    "image_count": len(cache_df),
    "source_archive": str(archive_path),
}
with open(EXPERIMENT_DIR / "sugarcane_cache_config.json", "w") as f:
    json.dump(cache_config, f, indent=2)

print(f"Cached images: {len(cache_df)}")
print(f"Colab cache: {CACHE_DIR}")
print(f"Cache manifest saved to: {cache_manifest_path}")

Caching Sugarcane images:   0%|          | 0/2240 [00:00<?, ?it/s]

Cached images: 2240
Colab cache: /content/sugarcane_cache_448
Cache manifest saved to: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment/sugarcane_resized_cache_manifest.csv


In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)
CACHE_DIR = Path("/content/sugarcane_cache_448")
CACHE_MANIFEST = EXPERIMENT_DIR / "sugarcane_resized_cache_manifest.csv"

IMAGE_SIZE = 448
TILE_SIZE = 288
BATCH_SIZE = 4
NUM_WORKERS = 0  # Stable for Colab and Drive-backed notebooks

CLASS_NAMES = ["Healthy", "Mosaic", "RedRot", "Rust", "Yellow"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

cache_df = pd.read_csv(CACHE_MANIFEST)
original_to_cached = dict(
    zip(cache_df["image_path"].astype(str), cache_df["cached_path"].astype(str))
)

train_df = pd.read_csv(EXPERIMENT_DIR / "train_manifest.csv")
val_df = pd.read_csv(EXPERIMENT_DIR / "validation_manifest.csv")
test_df = pd.read_csv(EXPERIMENT_DIR / "internal_test_manifest.csv")

for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    missing_labels = sorted(set(frame["label"]) - set(CLASS_TO_IDX))
    missing_cache = [p for p in frame["image_path"].astype(str) if p not in original_to_cached]
    if missing_labels or missing_cache:
        raise ValueError(
            f"{name}: missing labels={missing_labels}; uncached paths={len(missing_cache)}"
        )

# Four overlapping tiles, matching the established 448 / 288 input pipeline.
tile_starts = [
    (0, 0),
    (IMAGE_SIZE - TILE_SIZE, 0),
    (0, IMAGE_SIZE - TILE_SIZE),
    (IMAGE_SIZE - TILE_SIZE, IMAGE_SIZE - TILE_SIZE),
]

class SugarcaneDataset(Dataset):
    def __init__(self, dataframe, training=False):
        self.dataframe = dataframe.reset_index(drop=True)
        self.training = training

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        cached_path = original_to_cached[str(row["image_path"])]

        with Image.open(cached_path) as image_file:
            image = image_file.convert("RGB")

        # Apply the same random flip to full image and tiles to keep views aligned.
        if self.training and torch.rand(()) < 0.5:
            image = TF.hflip(image)

        full = TF.normalize(TF.to_tensor(image), MEAN, STD)

        tiles = []
        for left, top in tile_starts:
            tile = image.crop((left, top, left + TILE_SIZE, top + TILE_SIZE))
            tiles.append(TF.normalize(TF.to_tensor(tile), MEAN, STD))

        return (
            full,
            torch.stack(tiles),
            CLASS_TO_IDX[row["label"]],
        )

train_dataset = SugarcaneDataset(train_df, training=True)
val_dataset = SugarcaneDataset(val_df, training=False)
test_dataset = SugarcaneDataset(test_df, training=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

full_batch, tile_batch, label_batch = next(iter(train_loader))
print("Classes:", CLASS_NAMES)
print("Train / validation / test:", len(train_dataset), len(val_dataset), len(test_dataset))
print("Full batch:", tuple(full_batch.shape))
print("Tile batch:", tuple(tile_batch.shape))
print("Label batch:", tuple(label_batch.shape))
print("GPU available:", torch.cuda.is_available())

Classes: ['Healthy', 'Mosaic', 'RedRot', 'Rust', 'Yellow']
Train / validation / test: 1568 336 336
Full batch: (4, 3, 448, 448)
Tile batch: (4, 4, 3, 288, 288)
Label batch: (4,)
GPU available: True


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights

class SpatialSelfAttention(nn.Module):
    def __init__(self, channels=256, heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.attention = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=heads,
            batch_first=True,
        )
        self.gate = nn.Conv2d(channels * 2, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, height, width = x.shape

        tokens = x.flatten(2).transpose(1, 2)
        normalized = self.norm(tokens)
        attended, _ = self.attention(
            normalized,
            normalized,
            normalized,
            need_weights=False,
        )
        attended = attended.transpose(1, 2).reshape(
            batch, channels, height, width
        )

        gate = torch.sigmoid(self.gate(torch.cat([x, attended], dim=1)))
        return x + gate * attended


class SugarcaneResNet50Attention(nn.Module):
    def __init__(self, num_classes=5, pretrained=True):
        super().__init__()

        weights = ResNet50_Weights.DEFAULT if pretrained else None
        backbone = resnet50(weights=weights)

        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.project2 = nn.Conv2d(512, 256, kernel_size=1)
        self.project3 = nn.Conv2d(1024, 256, kernel_size=1)
        self.project4 = nn.Conv2d(2048, 256, kernel_size=1)

        self.fusion = nn.Sequential(
            nn.Conv2d(768, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.self_attention = SpatialSelfAttention(channels=256, heads=8)
        self.pool = nn.AdaptiveAvgPool2d(1)

        self.tile_score = nn.Linear(256, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.35),
            nn.Linear(768, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(512, num_classes),
        )

    def encode_view(self, images):
        x = self.stem(images)
        x = self.layer1(x)

        fine = self.layer2(x)
        middle = self.layer3(fine)
        deep = self.layer4(middle)

        output_size = fine.shape[-2:]
        fine = self.project2(fine)
        middle = F.interpolate(
            self.project3(middle),
            size=output_size,
            mode="bilinear",
            align_corners=False,
        )
        deep = F.interpolate(
            self.project4(deep),
            size=output_size,
            mode="bilinear",
            align_corners=False,
        )

        fused = self.fusion(torch.cat([fine, middle, deep], dim=1))
        attended = self.self_attention(fused)
        return self.pool(attended).flatten(1)

    def forward(self, full_images, tile_images):
        full_features = self.encode_view(full_images)

        batch_size, tile_count, channels, height, width = tile_images.shape
        flat_tiles = tile_images.reshape(
            batch_size * tile_count, channels, height, width
        )
        tile_features = self.encode_view(flat_tiles).reshape(
            batch_size, tile_count, -1
        )

        tile_weights = torch.softmax(
            self.tile_score(tile_features).squeeze(-1),
            dim=1,
        )
        weighted_tiles = (
            tile_features * tile_weights.unsqueeze(-1)
        ).sum(dim=1)
        strongest_tile = tile_features.max(dim=1).values

        combined = torch.cat(
            [full_features, weighted_tiles, strongest_tile],
            dim=1,
        )
        return self.classifier(combined)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SugarcaneResNet50Attention(
    num_classes=len(CLASS_NAMES),
    pretrained=True,
).to(DEVICE)

full_batch, tile_batch, label_batch = next(iter(train_loader))
full_batch = full_batch.to(DEVICE)
tile_batch = tile_batch.to(DEVICE)

model.eval()
with torch.inference_mode():
    output = model(full_batch, tile_batch)

print("Device:", DEVICE)
print("Output shape:", tuple(output.shape))
print("Output finite:", torch.isfinite(output).all().item())
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 65.0MB/s]


Device: cuda
Output shape: (4, 5)
Output finite: True
Trainable parameters: 26989382


In [17]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from torch.amp import GradScaler, autocast
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)
BEST_CHECKPOINT_PATH = EXPERIMENT_DIR / "sugarcane_best_model.pth"
LAST_CHECKPOINT_PATH = EXPERIMENT_DIR / "sugarcane_last_checkpoint.pth"
HISTORY_PATH = EXPERIMENT_DIR / "sugarcane_training_history.csv"

EPOCHS = 20
PATIENCE = 8
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
USE_AMP = torch.cuda.is_available()

# Weighted sampling gives the smaller classes more chances during training.
class_counts = train_df["label"].value_counts()
sample_weights = train_df["label"].map(
    lambda label: 1.0 / class_counts[label]
).to_numpy(dtype=np.float64)

sampler = torch.utils.data.WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_df),
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
# Keep validation deterministic and unweighted.
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)
scaler = GradScaler("cuda", enabled=USE_AMP)

start_epoch = 1
best_val_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0
history = []

# Resume training when a valid latest checkpoint exists.
if LAST_CHECKPOINT_PATH.exists():
    checkpoint = torch.load(
        LAST_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    if checkpoint.get("scaler_state_dict"):
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_val_f1 = checkpoint.get("best_val_f1", -1.0)
    best_epoch = checkpoint.get("best_epoch", 0)
    epochs_without_improvement = checkpoint.get("epochs_without_improvement", 0)
    history = checkpoint.get("history", [])

    print(f"Resuming from epoch {start_epoch}; best validation macro-F1={best_val_f1:.4f}")
else:
    print("Starting a fresh Sugarcane training run.")

def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    y_true, y_pred = [], []

    progress = tqdm(loader, desc="Training" if training else "Validation")
    for full_images, tile_images, targets in progress:
        full_images = full_images.to(DEVICE, non_blocking=True)
        tile_images = tile_images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with autocast(device_type="cuda", enabled=USE_AMP):
                logits = model(full_images, tile_images)
                loss = criterion(logits, targets)

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * targets.size(0)
        predictions = logits.argmax(dim=1)
        y_true.extend(targets.detach().cpu().tolist())
        y_pred.extend(predictions.detach().cpu().tolist())
        progress.set_postfix(loss=f"{loss.item():.4f}")

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }

for epoch in range(start_epoch, EPOCHS + 1):
    learning_rate = optimizer.param_groups[0]["lr"]
    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader, training=False)
    scheduler.step(val_metrics["macro_f1"])

    row = {
        "epoch": epoch,
        "learning_rate": learning_rate,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(row)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss={train_metrics['loss']:.4f}, "
        f"acc={train_metrics['accuracy']:.4f}, "
        f"macro-F1={train_metrics['macro_f1']:.4f} | "
        f"Val loss={val_metrics['loss']:.4f}, "
        f"acc={val_metrics['accuracy']:.4f}, "
        f"balanced acc={val_metrics['balanced_accuracy']:.4f}, "
        f"macro-F1={val_metrics['macro_f1']:.4f}"
    )

    improved = val_metrics["macro_f1"] > best_val_f1
    if improved:
        best_val_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    checkpoint_data = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_f1": best_val_f1,
        "best_epoch": best_epoch,
        "epochs_without_improvement": epochs_without_improvement,
        "class_names": CLASS_NAMES,
        "history": history,
        "metrics": row,
    }
    torch.save(checkpoint_data, LAST_CHECKPOINT_PATH)

    if improved:
        torch.save(checkpoint_data, BEST_CHECKPOINT_PATH)
        print(f"Saved new best checkpoint (epoch {epoch}).")
    else:
        print("Saved latest resumable checkpoint.")

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after {PATIENCE} epochs without validation macro-F1 improvement.")
        break

print("\nTraining finished.")
print(f"Best validation macro-F1: {best_val_f1:.4f} at epoch {best_epoch}")
print(f"Best checkpoint: {BEST_CHECKPOINT_PATH}")
print(f"Latest resumable checkpoint: {LAST_CHECKPOINT_PATH}")
print(f"Training history: {HISTORY_PATH}")

Resuming from epoch 20; best validation macro-F1=0.9833


Training:   0%|          | 0/392 [00:00<?, ?it/s]

Validation:   0%|          | 0/84 [00:00<?, ?it/s]

Epoch 20/20 | Train loss=0.0471, acc=0.9898, macro-F1=0.9897 | Val loss=0.0555, acc=0.9792, balanced acc=0.9792, macro-F1=0.9801
Saved latest resumable checkpoint.
Early stopping after 8 epochs without validation macro-F1 improvement.

Training finished.
Best validation macro-F1: 0.9833 at epoch 11
Best checkpoint: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment/sugarcane_best_model.pth
Latest resumable checkpoint: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment/sugarcane_last_checkpoint.pth
Training history: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment/sugarcane_training_history.csv


In [18]:
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)
BEST_CHECKPOINT_PATH = EXPERIMENT_DIR / "sugarcane_best_model.pth"

checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.to(DEVICE)
model.eval()

y_true, y_pred, y_prob = [], [], []

with torch.inference_mode():
    for full_images, tile_images, targets in tqdm(test_loader, desc="Sugarcane internal test"):
        full_images = full_images.to(DEVICE, non_blocking=True)
        tile_images = tile_images.to(DEVICE, non_blocking=True)

        logits = model(full_images, tile_images)
        probabilities = torch.softmax(logits, dim=1)
        predictions = probabilities.argmax(dim=1)

        y_true.extend(targets.tolist())
        y_pred.extend(predictions.cpu().tolist())
        y_prob.extend(probabilities.cpu().tolist())

accuracy = accuracy_score(y_true, y_pred)
balanced_acc = balanced_accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

report = classification_report(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))

print(f"Best checkpoint epoch: {checkpoint.get('epoch', 'unknown')}")
print(f"Internal test images: {len(y_true)}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro-F1: {macro_f1:.4f}")

print("\nPer-class report:")
display(pd.DataFrame(report).T.round(4))

print("\nConfusion matrix (true rows, predicted columns):")
display(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES))

# Save a detailed report, confusion matrix, and per-image predictions.
pd.DataFrame(report).T.to_csv(
    EXPERIMENT_DIR / "sugarcane_internal_test_classification_report.csv"
)
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
    EXPERIMENT_DIR / "sugarcane_internal_test_confusion_matrix.csv"
)

predictions_df = pd.read_csv(EXPERIMENT_DIR / "internal_test_manifest.csv").copy()
predictions_df["true_index"] = y_true
predictions_df["predicted_index"] = y_pred
predictions_df["predicted_label"] = [CLASS_NAMES[i] for i in y_pred]
predictions_df["confidence"] = [float(max(row)) for row in y_prob]
predictions_df["correct"] = predictions_df["true_index"] == predictions_df["predicted_index"]
predictions_df.to_csv(
    EXPERIMENT_DIR / "sugarcane_internal_test_predictions.csv",
    index=False,
)

metrics = {
    "checkpoint": str(BEST_CHECKPOINT_PATH),
    "best_epoch": checkpoint.get("epoch"),
    "accuracy": accuracy,
    "balanced_accuracy": balanced_acc,
    "macro_f1": macro_f1,
    "test_images": len(y_true),
    "class_names": CLASS_NAMES,
}
with open(EXPERIMENT_DIR / "sugarcane_internal_test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("\nSaved test evaluation files to:", EXPERIMENT_DIR)

Sugarcane internal test:   0%|          | 0/84 [00:00<?, ?it/s]

Best checkpoint epoch: 11
Internal test images: 336
Accuracy: 0.9762
Balanced accuracy: 0.9776
Macro-F1: 0.9763

Per-class report:


,precision,recall,f1-score,support
Healthy,0.9718,1.0000,0.9857,69.0000
Mosaic,0.9655,1.0000,0.9825,56.0000
RedRot,1.0000,0.9595,0.9793,74.0000
Rust,0.9559,0.9848,0.9701,66.0000
Yellow,0.9853,0.9437,0.9640,71.0000
accuracy,0.9762,0.9762,0.9762,0.9762
macro avg,0.9757,0.9776,0.9763,336.0000
weighted avg,0.9767,0.9762,0.9761,336.0000



Confusion matrix (true rows, predicted columns):


,Healthy,Mosaic,RedRot,Rust,Yellow
Healthy,69,0,0,0,0
Mosaic,0,56,0,0,0
RedRot,2,0,71,1,0
Rust,0,0,0,65,1
Yellow,0,2,0,2,67



Saved test evaluation files to: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_model_experiment


In [19]:
from pathlib import Path
import pandas as pd

TEA_OOD_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Tea/tea_ood_data"
)
SUGARCANE_OOD_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data"
)
SUGARCANE_OOD_DIR.mkdir(parents=True, exist_ok=True)

files_to_copy = {
    "plantdoc_inventory_manifest.csv":
        "plantdoc_inventory_manifest.csv",
    "plantdoc_ood_train_manifest.csv":
        "plantdoc_ood_train_manifest.csv",
    "plantdoc_ood_calibration_manifest.csv":
        "plantdoc_ood_calibration_manifest.csv",
    "plantdoc_ood_validation_manifest.csv":
        "plantdoc_ood_validation_manifest.csv",
    "plantdoc_ood_test_manifest.csv":
        "plantdoc_ood_test_manifest.csv",
}

missing_files = [
    filename for filename in files_to_copy
    if not (TEA_OOD_DIR / filename).exists()
]
if missing_files:
    raise FileNotFoundError(
        "These expected PlantDoc files are missing from the Tea OOD folder:\n"
        + "\n".join(missing_files)
    )

copied = {}
for source_name, destination_name in files_to_copy.items():
    source_path = TEA_OOD_DIR / source_name
    destination_path = SUGARCANE_OOD_DIR / destination_name

    # Copy the CSV contents without changing the originals.
    frame = pd.read_csv(source_path)
    frame.to_csv(destination_path, index=False)
    copied[destination_name] = frame

print("Copied PlantDoc manifests to:", SUGARCANE_OOD_DIR)

for filename, frame in copied.items():
    print(f"{filename}: {len(frame)} rows")

# Verify that the image paths in each split still exist.
path_column = "image_path"
for filename in [
    "plantdoc_ood_train_manifest.csv",
    "plantdoc_ood_calibration_manifest.csv",
    "plantdoc_ood_validation_manifest.csv",
    "plantdoc_ood_test_manifest.csv",
]:
    frame = copied[filename]
    if path_column not in frame.columns:
        print(f"{filename}: no '{path_column}' column; columns={list(frame.columns)}")
        continue

    paths = frame[path_column].dropna().astype(str)
    missing_paths = [path for path in paths if not Path(path).exists()]

    print(
        f"{filename}: {len(paths) - len(missing_paths)}/{len(paths)} "
        "image paths currently resolve"
    )
    if missing_paths:
        print("  Missing examples:", missing_paths[:3])

print("\nOOD files now in Sugarcane folder:")
for path in sorted(SUGARCANE_OOD_DIR.glob("plantdoc_*.csv")):
    print("-", path.name)

Copied PlantDoc manifests to: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data
plantdoc_inventory_manifest.csv: 2578 rows
plantdoc_ood_train_manifest.csv: 1631 rows
plantdoc_ood_calibration_manifest.csv: 349 rows
plantdoc_ood_validation_manifest.csv: 349 rows
plantdoc_ood_test_manifest.csv: 228 rows
plantdoc_ood_train_manifest.csv: 1631/1631 image paths currently resolve
plantdoc_ood_calibration_manifest.csv: 349/349 image paths currently resolve
plantdoc_ood_validation_manifest.csv: 349/349 image paths currently resolve
plantdoc_ood_test_manifest.csv: 228/228 image paths currently resolve

OOD files now in Sugarcane folder:
- plantdoc_inventory_manifest.csv
- plantdoc_ood_calibration_manifest.csv
- plantdoc_ood_test_manifest.csv
- plantdoc_ood_train_manifest.csv
- plantdoc_ood_validation_manifest.csv


In [20]:
import hashlib
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)
OOD_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_ood_data"
)

sugarcane_files = [
    EXPERIMENT_DIR / "train_manifest.csv",
    EXPERIMENT_DIR / "validation_manifest.csv",
    EXPERIMENT_DIR / "internal_test_manifest.csv",
]
plantdoc_files = [
    OOD_DIR / "plantdoc_ood_train_manifest.csv",
    OOD_DIR / "plantdoc_ood_calibration_manifest.csv",
    OOD_DIR / "plantdoc_ood_validation_manifest.csv",
    OOD_DIR / "plantdoc_ood_test_manifest.csv",
]

for path in sugarcane_files + plantdoc_files:
    if not path.exists():
        raise FileNotFoundError(f"Required manifest not found: {path}")

sugarcane_df = pd.concat(
    [pd.read_csv(path) for path in sugarcane_files],
    ignore_index=True,
).drop_duplicates("image_path")

plantdoc_df = pd.concat(
    [pd.read_csv(path) for path in plantdoc_files],
    ignore_index=True,
).drop_duplicates("image_path")

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def add_hashes(frame, description):
    hashes = []
    missing = []
    for path in tqdm(frame["image_path"].astype(str), desc=description):
        if not Path(path).is_file():
            hashes.append(None)
            missing.append(path)
        else:
            hashes.append(sha256_file(path))
    result = frame.copy()
    result["sha256"] = hashes
    return result, missing

sugarcane_hashed, sugarcane_missing = add_hashes(
    sugarcane_df, "Hashing Sugarcane images"
)
plantdoc_hashed, plantdoc_missing = add_hashes(
    plantdoc_df, "Hashing PlantDoc images"
)

sugarcane_hashes = set(sugarcane_hashed["sha256"].dropna())
plantdoc_hashed["matches_sugarcane"] = plantdoc_hashed["sha256"].isin(sugarcane_hashes)

matches = plantdoc_hashed[plantdoc_hashed["matches_sugarcane"]].merge(
    sugarcane_hashed[["image_path", "label", "sha256"]],
    on="sha256",
    how="inner",
    suffixes=("_plantdoc", "_sugarcane"),
)

match_path = OOD_DIR / "plantdoc_sugarcane_exact_duplicate_matches.csv"
plantdoc_hashed.to_csv(OOD_DIR / "plantdoc_hashed_manifest.csv", index=False)
sugarcane_hashed.to_csv(OOD_DIR / "sugarcane_hashed_manifest.csv", index=False)
matches.to_csv(match_path, index=False)

print("Sugarcane images hashed:", sugarcane_hashed["sha256"].notna().sum())
print("PlantDoc images hashed:", plantdoc_hashed["sha256"].notna().sum())
print("Missing Sugarcane paths:", len(sugarcane_missing))
print("Missing PlantDoc paths:", len(plantdoc_missing))
print("Exact duplicate matches:", len(matches))
print("Match report saved to:", match_path)

if not matches.empty:
    display(matches[[
        "image_path_plantdoc",
        "plantdoc_class",
        "ood_split",
        "image_path_sugarcane",
        "label",
    ]].head(30))

Hashing Sugarcane images:   0%|          | 0/2240 [00:00<?, ?it/s]

Hashing PlantDoc images:   0%|          | 0/2557 [00:00<?, ?it/s]

Sugarcane images hashed: 2240
PlantDoc images hashed: 2557
Missing Sugarcane paths: 0
Missing PlantDoc paths: 0
Exact duplicate matches: 0
Match report saved to: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/plantdoc_sugarcane_exact_duplicate_matches.csv


In [21]:
import pandas as pd
import torch
from pathlib import Path
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF

OOD_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_ood_data"
)
OOD_TRAIN_CSV = OOD_DIR / "plantdoc_ood_train_manifest.csv"

if "CLASS_NAMES" not in globals() or "DEVICE" not in globals():
    raise RuntimeError("Sugarcane model setup is missing. Run the Sugarcane loader and model cells first.")
if "make_views" in globals():
    print("Using existing make_views function.")
else:
    print("Defining matching 448 full-image and four-tile preprocessing.")

ood_train_df = pd.read_csv(OOD_TRAIN_CSV)
if not {"image_path", "plantdoc_class"}.issubset(ood_train_df.columns):
    raise ValueError(f"Unexpected PlantDoc manifest columns: {list(ood_train_df.columns)}")

missing = [p for p in ood_train_df["image_path"].astype(str) if not Path(p).is_file()]
if missing:
    raise FileNotFoundError(f"{len(missing)} OOD image files are missing. Examples: {missing[:5]}")

# Match the Sugarcane model's normalization and image geometry.
IMAGE_SIZE = 448
TILE_SIZE = 288
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
TILE_STARTS = [
    (0, 0),
    (IMAGE_SIZE - TILE_SIZE, 0),
    (0, IMAGE_SIZE - TILE_SIZE),
    (IMAGE_SIZE - TILE_SIZE, IMAGE_SIZE - TILE_SIZE),
]

def make_ood_views(image_path):
    with Image.open(image_path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
        image = ImageOps.pad(
            image,
            (IMAGE_SIZE, IMAGE_SIZE),
            method=Image.Resampling.LANCZOS,
            color=(127, 127, 127),
        )

    full = TF.normalize(TF.to_tensor(image), MEAN, STD)
    tiles = []
    for left, top in TILE_STARTS:
        tile = image.crop((left, top, left + TILE_SIZE, top + TILE_SIZE))
        tiles.append(TF.normalize(TF.to_tensor(tile), MEAN, STD))
    return full, torch.stack(tiles)

class PlantDocOODDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        path = self.dataframe.iloc[index]["image_path"]
        full, tiles = make_ood_views(path)
        return full, tiles

OOD_BATCH_SIZE = 4
ood_train_dataset = PlantDocOODDataset(ood_train_df)
ood_train_loader = DataLoader(
    ood_train_dataset,
    batch_size=OOD_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

ood_full_batch, ood_tile_batch = next(iter(ood_train_loader))
print("PlantDoc OOD training images:", len(ood_train_dataset))
print("OOD full batch:", tuple(ood_full_batch.shape))
print("OOD tile batch:", tuple(ood_tile_batch.shape))
print("OOD batch values finite:", torch.isfinite(ood_full_batch).all().item()
      and torch.isfinite(ood_tile_batch).all().item())

Defining matching 448 full-image and four-tile preprocessing.
PlantDoc OOD training images: 1631
OOD full batch: (4, 3, 448, 448)
OOD tile batch: (4, 4, 3, 288, 288)
OOD batch values finite: True


In [22]:
import pandas as pd
from pathlib import Path

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)

SPLIT_FILES = {
    "train": EXPERIMENT_DIR / "train_manifest.csv",
    "validation": EXPERIMENT_DIR / "validation_manifest.csv",
    "internal_test": EXPERIMENT_DIR / "internal_test_manifest.csv",
}
PAIR_FILES = [
    EXPERIMENT_DIR / "sugarcane_near_duplicate_review_subset.csv",
    EXPERIMENT_DIR / "sugarcane_near_duplicate_review_next_60.csv",
]

split_frames = {}
path_to_split = {}

for split_name, csv_path in SPLIT_FILES.items():
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing split manifest: {csv_path}")
    frame = pd.read_csv(csv_path)
    split_frames[split_name] = frame
    for image_path in frame["image_path"].astype(str):
        path_to_split[image_path] = split_name

print("Current split sizes:")
for split_name, frame in split_frames.items():
    print(f"  {split_name}: {len(frame)}")

checked_pairs = 0
cross_split_pairs = []

for pair_file in PAIR_FILES:
    if not pair_file.exists():
        print(f"Pair file not found, skipping: {pair_file.name}")
        continue

    pairs = pd.read_csv(pair_file)
    required = {"image_path_1", "image_path_2"}
    if not required.issubset(pairs.columns):
        print(f"Unexpected columns in {pair_file.name}: {list(pairs.columns)}")
        continue

    for _, row in pairs.iterrows():
        first = str(row["image_path_1"])
        second = str(row["image_path_2"])
        first_split = path_to_split.get(first)
        second_split = path_to_split.get(second)

        if first_split is None or second_split is None:
            continue

        checked_pairs += 1
        if first_split != second_split:
            cross_split_pairs.append({
                "pair_file": pair_file.name,
                "image_path_1": first,
                "split_1": first_split,
                "image_path_2": second,
                "split_2": second_split,
            })

print(f"\nReviewed pairs found in current splits: {checked_pairs}")
print(f"Reviewed pairs crossing splits: {len(cross_split_pairs)}")

if cross_split_pairs:
    print("\nExamples of cross-split pairs:")
    display(pd.DataFrame(cross_split_pairs).head(20))
else:
    print("No reviewed pair was found crossing the current splits.")


Current split sizes:
  train: 1568
  validation: 336
  internal_test: 336

Reviewed pairs found in current splits: 50
Reviewed pairs crossing splits: 17

Examples of cross-split pairs:


,pair_file,image_path_1,split_1,image_path_2,split_2
0,sugarcane_near_duplicate_review_next_60.csv,/content/sugarcane_raw/Sugarcane Leaf Disease ...,validation,/content/sugarcane_raw/Sugarcane Leaf Disease ...,train
1,sugarcane_near_duplicate_review_next_60.csv,/content/sugarcane_raw/Sugarcane Leaf Disease ...,validation,/content/sugarcane_raw/Sugarcane Leaf Disease ...,train
2,sugarcane_near_duplicate_review_next_60.csv,/content/sugarcane_raw/Sugarcane Leaf Disease ...,train,/content/sugarcane_raw/Sugarcane Leaf Disease ...,internal_test
3,sugarcane_near_duplicate_review_next_60.csv,/content/sugarcane_raw/Sugarcane Leaf Disease ...,train,/content/sugarcane_raw/Sugarcane Leaf Disease ...,validation
4,sugarcane_near_duplicate_review_next_60.csv,/content/sugarcane_raw/Sugarcane Leaf Disease ...,internal_test,/content/sugarcane_raw/Sugarcane Leaf Disease ...,train
5,sugarcane_near_duplicate_review_next_60.csv,/content/sugarcane_raw/Sugarcane Leaf Disease ...,train,/content/sugarcane_raw/Sugarcane Leaf Disease ...,validation
6,sugarcane_near_duplicate_review_next_60.csv,/content/sugarcane_raw/Sugarcane Leaf Disease ...,validation,/content/sugarcane_raw/Sugarcane Leaf Disease ...,train
7,sugarcane_near_duplicate_review_next_60.csv,/content/sugarcane_raw/Sugarcane Leaf Disease ...,internal_test,/content/sugarcane_raw/Sugarcane Leaf Disease ...,train
8,sugarcane_near_duplicate_review_next_60.csv,/content/sugarcane_raw/Sugarcane Leaf Disease ...,internal_test,/content/sugarcane_raw/Sugarcane Leaf Disease ...,train
9,sugarcane_near_duplicate_review_next_60.csv,/content/sugarcane_raw/Sugarcane Leaf Disease ...,train,/content/sugarcane_raw/Sugarcane Leaf Disease ...,internal_test


In [23]:
import shutil
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import DataLoader

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)
OOD_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_ood_data"
)
CHECKPOINT_PATH = EXPERIMENT_DIR / "sugarcane_best_model.pth"
BASELINE_COPY_PATH = OOD_DIR / "sugarcane_baseline_before_ood.pth"

# Preserve an untouched copy of the supervised classifier checkpoint.
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Best Sugarcane checkpoint not found: {CHECKPOINT_PATH}")

if not BASELINE_COPY_PATH.exists():
    shutil.copy2(CHECKPOINT_PATH, BASELINE_COPY_PATH)

# Load Sugarcane training data only. The validation and test manifests are not used here.
sugarcane_train_df = pd.read_csv(EXPERIMENT_DIR / "train_manifest.csv")
plantdoc_train_df = pd.read_csv(OOD_DIR / "plantdoc_ood_train_manifest.csv")

needed_columns = {"image_path", "label"}
if not needed_columns.issubset(sugarcane_train_df.columns):
    raise ValueError(f"Sugarcane manifest columns: {list(sugarcane_train_df.columns)}")
if "image_path" not in plantdoc_train_df.columns:
    raise ValueError(f"PlantDoc manifest columns: {list(plantdoc_train_df.columns)}")

# Use the same view-generation function and dimensions as the supervised pipeline.
# SugarcaneDataset returns (full, tiles, target), so PlantDoc needs an unlabeled wrapper.
class SugarcaneOODDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        image_path = self.dataframe.iloc[index]["image_path"]
        full, tiles = make_ood_views(image_path)
        return full, tiles

# Check that loaders, paths, and model setup exist from prior cells.
required_names = ["train_dataset", "make_ood_views", "DEVICE", "model", "CLASS_NAMES"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError(
        "These Sugarcane setup objects are missing from the current runtime: "
        + ", ".join(missing_names)
        + ". Re-run the corresponding setup cells first."
    )

# Use the existing Sugarcane dataset class for labeled in-distribution batches.
sugarcane_train_dataset = train_dataset
plantdoc_ood_dataset = SugarcaneOODDataset(plantdoc_train_df)

BATCH_SIZE = 4
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

sugarcane_train_loader = DataLoader(
    sugarcane_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True,
)
plantdoc_ood_train_loader = DataLoader(
    plantdoc_ood_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True,
)

# Confirm the source data and model inputs before training.
sugar_full, sugar_tiles, sugar_targets = next(iter(sugarcane_train_loader))
ood_full, ood_tiles = next(iter(plantdoc_ood_train_loader))

print("Untouched baseline checkpoint:", BASELINE_COPY_PATH)
print("Sugarcane training images:", len(sugarcane_train_dataset))
print("PlantDoc OOD training images:", len(plantdoc_ood_dataset))
print("Sugarcane batch:", tuple(sugar_full.shape), tuple(sugar_tiles.shape), tuple(sugar_targets.shape))
print("PlantDoc batch:", tuple(ood_full.shape), tuple(ood_tiles.shape))
print("Internal test manifest remains unused in this cell.")

Untouched baseline checkpoint: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/sugarcane_baseline_before_ood.pth
Sugarcane training images: 1568
PlantDoc OOD training images: 1631
Sugarcane batch: (4, 3, 448, 448) (4, 4, 3, 288, 288) (4,)
PlantDoc batch: (4, 3, 448, 448) (4, 4, 3, 288, 288)
Internal test manifest remains unused in this cell.


In [24]:
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from torch.amp import GradScaler, autocast
from tqdm.auto import tqdm

OOD_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_ood_data"
)
EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)

BASELINE_PATH = OOD_DIR / "sugarcane_baseline_before_ood.pth"
LAST_OE_PATH = OOD_DIR / "sugarcane_oe_last.pth"
BEST_OE_PATH = OOD_DIR / "sugarcane_oe_best.pth"
HISTORY_PATH = OOD_DIR / "sugarcane_oe_training_history.csv"

EPOCHS = 3
LEARNING_RATE = 1e-6
OOD_LOSS_WEIGHT = 0.5
USE_AMP = torch.cuda.is_available()

if not BASELINE_PATH.exists():
    raise FileNotFoundError(f"Baseline checkpoint not found: {BASELINE_PATH}")

if "SugarcaneResNet50Attention" not in globals():
    raise RuntimeError("SugarcaneResNet50Attention is not defined. Run the model-definition cell.")
if "sugarcane_train_loader" not in globals() or "plantdoc_ood_train_loader" not in globals():
    raise RuntimeError("Training loaders are missing. Run the OOD loader setup cell first.")

# Reload a fresh model from the untouched supervised checkpoint.
baseline = torch.load(BASELINE_PATH, map_location=DEVICE, weights_only=False)
model = SugarcaneResNet50Attention(
    num_classes=len(CLASS_NAMES),
    pretrained=False,
).to(DEVICE)
model.load_state_dict(baseline["model_state_dict"], strict=True)
model.eval()

# Fine-tune the classifier and final ResNet stage; keep lower visual features fixed.
for parameter in model.parameters():
    parameter.requires_grad = False

for parameter in model.layer4.parameters():
    parameter.requires_grad = True
for parameter in model.classifier.parameters():
    parameter.requires_grad = True
for parameter in model.tile_score.parameters():
    parameter.requires_grad = True
for parameter in model.project4.parameters():
    parameter.requires_grad = True
for parameter in model.fusion.parameters():
    parameter.requires_grad = True
for parameter in model.self_attention.parameters():
    parameter.requires_grad = True

optimizer = torch.optim.AdamW(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=1e-4,
)
scaler = GradScaler("cuda", enabled=USE_AMP)

start_epoch = 1
best_ood_loss = float("inf")
history = []

if LAST_OE_PATH.exists():
    checkpoint = torch.load(LAST_OE_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if checkpoint.get("scaler_state_dict"):
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_ood_loss = checkpoint.get("best_ood_loss", float("inf"))
    history = checkpoint.get("history", [])
    print(f"Resuming at epoch {start_epoch}.")
else:
    print("Starting OOD fine-tuning from the untouched Sugarcane baseline.")

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    sugar_iter = iter(sugarcane_train_loader)
    ood_iter = iter(plantdoc_ood_train_loader)
    steps = max(len(sugarcane_train_loader), len(plantdoc_ood_train_loader))

    total_sugar_loss = 0.0
    total_ood_loss = 0.0
    total_steps = 0
    progress = tqdm(range(steps), desc=f"Sugarcane OE epoch {epoch}/{EPOCHS}")

    for _ in progress:
        try:
            sugar_full, sugar_tiles, sugar_targets = next(sugar_iter)
        except StopIteration:
            sugar_iter = iter(sugarcane_train_loader)
            sugar_full, sugar_tiles, sugar_targets = next(sugar_iter)

        try:
            ood_full, ood_tiles = next(ood_iter)
        except StopIteration:
            ood_iter = iter(plantdoc_ood_train_loader)
            ood_full, ood_tiles = next(ood_iter)

        sugar_full = sugar_full.to(DEVICE, non_blocking=True)
        sugar_tiles = sugar_tiles.to(DEVICE, non_blocking=True)
        sugar_targets = sugar_targets.to(DEVICE, non_blocking=True)
        ood_full = ood_full.to(DEVICE, non_blocking=True)
        ood_tiles = ood_tiles.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type="cuda", enabled=USE_AMP):
            sugar_logits = model(sugar_full, sugar_tiles)
            ood_logits = model(ood_full, ood_tiles)

            # Keep correct Sugarcane classification.
            sugar_loss = F.cross_entropy(sugar_logits, sugar_targets)

            # Encourage a uniform class distribution on unsupported crop images.
            ood_log_probs = F.log_softmax(ood_logits, dim=1)
            ood_loss = F.kl_div(
                ood_log_probs,
                torch.full_like(ood_log_probs, 1.0 / len(CLASS_NAMES)),
                reduction="batchmean",
            )

            loss = sugar_loss + OOD_LOSS_WEIGHT * ood_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=1.0,
        )
        scaler.step(optimizer)
        scaler.update()

        total_sugar_loss += sugar_loss.item()
        total_ood_loss += ood_loss.item()
        total_steps += 1
        progress.set_postfix(
            sugar_loss=f"{sugar_loss.item():.3f}",
            ood_loss=f"{ood_loss.item():.3f}",
        )

    mean_sugar_loss = total_sugar_loss / total_steps
    mean_ood_loss = total_ood_loss / total_steps
    combined_loss = mean_sugar_loss + OOD_LOSS_WEIGHT * mean_ood_loss

    row = {
        "epoch": epoch,
        "sugarcane_train_loss": mean_sugar_loss,
        "plantdoc_ood_loss": mean_ood_loss,
        "combined_loss": combined_loss,
    }
    history.append(row)
    print(
        f"Epoch {epoch} | Sugarcane loss={mean_sugar_loss:.4f} | "
        f"PlantDoc uniformity loss={mean_ood_loss:.4f}"
    )

    checkpoint_data = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "class_names": CLASS_NAMES,
        "history": history,
        "metrics": row,
        "best_ood_loss": min(best_ood_loss, combined_loss),
        "ood_loss_weight": OOD_LOSS_WEIGHT,
        "learning_rate": LEARNING_RATE,
    }
    torch.save(checkpoint_data, LAST_OE_PATH)

    if combined_loss < best_ood_loss:
        best_ood_loss = combined_loss
        checkpoint_data["best_ood_loss"] = best_ood_loss
        torch.save(checkpoint_data, BEST_OE_PATH)
        print("Saved new best OOD checkpoint.")
    else:
        print("Saved resumable checkpoint.")

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

print("\nOOD fine-tuning finished.")
print("Best OOD checkpoint:", BEST_OE_PATH)
print("Latest resumable checkpoint:", LAST_OE_PATH)
print("Training history:", HISTORY_PATH)

Starting OOD fine-tuning from the untouched Sugarcane baseline.


Sugarcane OE epoch 1/3:   0%|          | 0/407 [00:00<?, ?it/s]

Epoch 1 | Sugarcane loss=0.0871 | PlantDoc uniformity loss=2.7662
Saved new best OOD checkpoint.


Sugarcane OE epoch 2/3:   0%|          | 0/407 [00:00<?, ?it/s]

Epoch 2 | Sugarcane loss=0.0728 | PlantDoc uniformity loss=2.0032
Saved new best OOD checkpoint.


Sugarcane OE epoch 3/3:   0%|          | 0/407 [00:00<?, ?it/s]

Epoch 3 | Sugarcane loss=0.0885 | PlantDoc uniformity loss=1.3783
Saved new best OOD checkpoint.

OOD fine-tuning finished.
Best OOD checkpoint: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/sugarcane_oe_best.pth
Latest resumable checkpoint: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/sugarcane_oe_last.pth
Training history: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/sugarcane_oe_training_history.csv


In [25]:
import json
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)
OOD_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_ood_data"
)
BEST_OE_PATH = OOD_DIR / "sugarcane_oe_best.pth"

if not BEST_OE_PATH.exists():
    raise FileNotFoundError(f"Best OOD checkpoint not found: {BEST_OE_PATH}")

# Load best OE weights into the already-defined matching architecture.
oe_checkpoint = torch.load(BEST_OE_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(oe_checkpoint["model_state_dict"], strict=True)
model.to(DEVICE).eval()

# Use your existing Sugarcane validation dataset, which returns labels.
if "val_dataset" not in globals():
    val_df = pd.read_csv(EXPERIMENT_DIR / "validation_manifest.csv")
    val_dataset = SugarcaneDataset(val_df, training=False)

supported_val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

def score_supported(loader, label):
    y_true, y_pred, max_probs = [], [], []
    with torch.inference_mode():
        for full, tiles, targets in tqdm(loader, desc=label):
            full = full.to(DEVICE, non_blocking=True)
            tiles = tiles.to(DEVICE, non_blocking=True)
            logits = model(full, tiles)
            probs = torch.softmax(logits, dim=1)
            y_true.extend(targets.tolist())
            y_pred.extend(probs.argmax(dim=1).cpu().tolist())
            max_probs.extend(probs.max(dim=1).values.cpu().tolist())

    metrics = {
        "images": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "mean_max_probability": float(sum(max_probs) / len(max_probs)),
    }
    print(f"\n{label}: {metrics}")
    return metrics, max_probs

def score_ood_manifest(filename, label):
    frame = pd.read_csv(OOD_DIR / filename)
    required = {"image_path"}
    if not required.issubset(frame.columns):
        raise ValueError(f"{filename} missing image_path column: {list(frame.columns)}")

    dataset = PlantDocOODDataset(frame)
    loader = DataLoader(
        dataset,
        batch_size=4,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    max_probs = []
    with torch.inference_mode():
        for full, tiles in tqdm(loader, desc=label):
            full = full.to(DEVICE, non_blocking=True)
            tiles = tiles.to(DEVICE, non_blocking=True)
            probs = torch.softmax(model(full, tiles), dim=1)
            max_probs.extend(probs.max(dim=1).values.cpu().tolist())

    result = {
        "images": len(max_probs),
        "mean_max_probability": float(sum(max_probs) / len(max_probs)),
        "median_max_probability": float(pd.Series(max_probs).median()),
        "p90_max_probability": float(pd.Series(max_probs).quantile(0.90)),
    }
    print(f"\n{label}: {result}")
    return result, max_probs

supported_metrics, supported_val_conf = score_supported(
    supported_val_loader, "Sugarcane supported validation"
)

plantdoc_cal_conf = {}
plantdoc_val_conf = {}
for filename, label, destination in [
    ("plantdoc_ood_calibration_manifest.csv", "PlantDoc calibration", plantdoc_cal_conf),
    ("plantdoc_ood_validation_manifest.csv", "PlantDoc validation", plantdoc_val_conf),
]:
    result, scores = score_ood_manifest(filename, label)
    destination["metrics"] = result
    destination["max_probability"] = scores

# Save raw validation confidence scores for later threshold calibration.
score_rows = []
for score in supported_val_conf:
    score_rows.append({
        "group": "supported_sugarcane_validation",
        "max_probability": score,
        "is_supported": True,
    })
for group_name, scores in [
    ("plantdoc_calibration", plantdoc_cal_conf["max_probability"]),
    ("plantdoc_validation", plantdoc_val_conf["max_probability"]),
]:
    for score in scores:
        score_rows.append({
            "group": group_name,
            "max_probability": score,
            "is_supported": False,
        })

scores_path = OOD_DIR / "sugarcane_oe_calibration_confidence_scores.csv"
pd.DataFrame(score_rows).to_csv(scores_path, index=False)

summary = {
    "checkpoint": str(BEST_OE_PATH),
    "checkpoint_epoch": oe_checkpoint.get("epoch"),
    "supported_validation": supported_metrics,
    "plantdoc_calibration": plantdoc_cal_conf["metrics"],
    "plantdoc_validation": plantdoc_val_conf["metrics"],
    "confidence_scores_csv": str(scores_path),
}
summary_path = OOD_DIR / "sugarcane_oe_validation_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved confidence scores:", scores_path)
print("Saved summary:", summary_path)

Sugarcane supported validation:   0%|          | 0/84 [00:00<?, ?it/s]


Sugarcane supported validation: {'images': 336, 'accuracy': 0.9285714285714286, 'balanced_accuracy': np.float64(0.9307265377687914), 'macro_f1': 0.9271625108808788, 'mean_max_probability': 0.9343066814222506}


PlantDoc calibration:   0%|          | 0/88 [00:00<?, ?it/s]


PlantDoc calibration: {'images': 349, 'mean_max_probability': 0.629444858406198, 'median_max_probability': 0.6092224717140198, 'p90_max_probability': 0.9222415685653687}


PlantDoc validation:   0%|          | 0/88 [00:00<?, ?it/s]


PlantDoc validation: {'images': 349, 'mean_max_probability': 0.6436843080472809, 'median_max_probability': 0.6083278059959412, 'p90_max_probability': 0.9532421946525573}

Saved confidence scores: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/sugarcane_oe_calibration_confidence_scores.csv
Saved summary: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/sugarcane_oe_validation_summary.json


In [26]:
import pandas as pd
from pathlib import Path

OOD_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_ood_data"
)
SCORES_PATH = OOD_DIR / "sugarcane_oe_calibration_confidence_scores.csv"

scores = pd.read_csv(SCORES_PATH)

supported = scores.loc[
    scores["group"] == "supported_sugarcane_validation",
    "max_probability",
].to_numpy()

ood_calibration = scores.loc[
    scores["group"] == "plantdoc_calibration",
    "max_probability",
].to_numpy()

if len(supported) == 0 or len(ood_calibration) == 0:
    raise ValueError("Missing supported or PlantDoc calibration confidence scores.")

# Accept an image when its top softmax probability >= threshold.
# Pick the highest threshold that retains at least 95% of supported images.
MIN_SUPPORTED_ACCEPTANCE = 0.95
candidate_thresholds = sorted(set(supported.tolist() + ood_calibration.tolist()))

rows = []
for threshold in candidate_thresholds:
    supported_acceptance = float((supported >= threshold).mean())
    plantdoc_rejection = float((ood_calibration < threshold).mean())
    rows.append({
        "threshold": threshold,
        "supported_acceptance": supported_acceptance,
        "plantdoc_rejection": plantdoc_rejection,
    })

sweep = pd.DataFrame(rows)
eligible = sweep[sweep["supported_acceptance"] >= MIN_SUPPORTED_ACCEPTANCE]

if eligible.empty:
    raise RuntimeError("No threshold meets the supported acceptance constraint.")

# Among eligible thresholds, maximize PlantDoc rejection.
# On ties, prefer the higher supported acceptance.
selected = eligible.sort_values(
    ["plantdoc_rejection", "supported_acceptance", "threshold"],
    ascending=[False, False, True],
).iloc[0]

print(f"Selected confidence threshold: {selected['threshold']:.6f}")
print(f"Supported acceptance on calibration: {selected['supported_acceptance']:.1%}")
print(f"PlantDoc rejection on calibration: {selected['plantdoc_rejection']:.1%}")

print("\nBest calibration trade-offs:")
display(
    eligible.sort_values("plantdoc_rejection", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

sweep_path = OOD_DIR / "sugarcane_confidence_threshold_sweep.csv"
sweep.to_csv(sweep_path, index=False)
print("\nSaved full sweep:", sweep_path)

Selected confidence threshold: 0.578150
Supported acceptance on calibration: 95.2%
PlantDoc rejection on calibration: 45.0%

Best calibration trade-offs:


,threshold,supported_acceptance,plantdoc_rejection
0,0.578150,0.952381,0.449857
1,0.573226,0.955357,0.446991
2,0.577277,0.952381,0.446991
3,0.570989,0.955357,0.444126
4,0.570441,0.955357,0.441261
5,0.569563,0.955357,0.438395
6,0.567784,0.955357,0.435530
7,0.566858,0.955357,0.432665
8,0.566045,0.955357,0.429799
9,0.564355,0.958333,0.426934



Saved full sweep: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/sugarcane_confidence_threshold_sweep.csv


In [27]:
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

OOD_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_ood_data"
)
EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)

THRESHOLD = 0.578150

# Evaluate PlantDoc validation rejection with the frozen calibration threshold.
plantdoc_val_df = pd.read_csv(OOD_DIR / "plantdoc_ood_validation_manifest.csv")
plantdoc_val_dataset = PlantDocOODDataset(plantdoc_val_df)
plantdoc_val_loader = DataLoader(
    plantdoc_val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

plantdoc_rows = []
model.eval()
with torch.inference_mode():
    for batch_index, (full, tiles) in enumerate(
        tqdm(plantdoc_val_loader, desc="PlantDoc validation threshold check")
    ):
        full = full.to(DEVICE, non_blocking=True)
        tiles = tiles.to(DEVICE, non_blocking=True)
        probabilities = torch.softmax(model(full, tiles), dim=1)
        confidence = probabilities.max(dim=1).values.cpu().tolist()

        start = batch_index * plantdoc_val_loader.batch_size
        batch_rows = plantdoc_val_df.iloc[start:start + len(confidence)]
        for (_, row), score in zip(batch_rows.iterrows(), confidence):
            plantdoc_rows.append({
                "image_path": row["image_path"],
                "plantdoc_class": row.get("plantdoc_class", "unknown"),
                "confidence": score,
                "accepted_as_sugarcane": score >= THRESHOLD,
                "rejected_as_ood": score < THRESHOLD,
            })

plantdoc_val_scored = pd.DataFrame(plantdoc_rows)
plantdoc_rejection = plantdoc_val_scored["rejected_as_ood"].mean()

print(f"Frozen confidence threshold: {THRESHOLD:.6f}")
print(f"PlantDoc validation images: {len(plantdoc_val_scored)}")
print(f"PlantDoc validation rejection: {plantdoc_rejection:.1%}")

print("\nPlantDoc validation rejection by class:")
display(
    plantdoc_val_scored.groupby("plantdoc_class")
    .agg(
        images=("image_path", "size"),
        rejected=("rejected_as_ood", "sum"),
        rejection_rate=("rejected_as_ood", "mean"),
        mean_confidence=("confidence", "mean"),
    )
    .sort_values("rejection_rate", ascending=False)
    .assign(rejection_rate=lambda table: (100 * table["rejection_rate"]).round(1))
    .assign(mean_confidence=lambda table: table["mean_confidence"].round(3))
)

# Check acceptance on Sugarcane validation, overall and by class.
sugar_val_df = pd.read_csv(EXPERIMENT_DIR / "validation_manifest.csv")
sugar_val_dataset = SugarcaneDataset(sugar_val_df, training=False)
sugar_val_loader = DataLoader(
    sugar_val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

sugar_rows = []
with torch.inference_mode():
    for batch_index, (full, tiles, targets) in enumerate(
        tqdm(sugar_val_loader, desc="Sugarcane validation threshold check")
    ):
        full = full.to(DEVICE, non_blocking=True)
        tiles = tiles.to(DEVICE, non_blocking=True)
        probabilities = torch.softmax(model(full, tiles), dim=1)
        confidence = probabilities.max(dim=1).values.cpu().tolist()
        predictions = probabilities.argmax(dim=1).cpu().tolist()

        start = batch_index * sugar_val_loader.batch_size
        batch_rows = sugar_val_df.iloc[start:start + len(confidence)]
        for offset, ((_, row), score, prediction) in enumerate(
            zip(batch_rows.iterrows(), confidence, predictions)
        ):
            sugar_rows.append({
                "image_path": row["image_path"],
                "label": row["label"],
                "confidence": score,
                "prediction": CLASS_NAMES[prediction],
                "accepted": score >= THRESHOLD,
            })

sugar_val_scored = pd.DataFrame(sugar_rows)
print(f"\nSugarcane validation accepted: {sugar_val_scored['accepted'].mean():.1%}")
print("Sugarcane validation acceptance by class:")
display(
    sugar_val_scored.groupby("label")
    .agg(
        images=("image_path", "size"),
        accepted=("accepted", "sum"),
        acceptance_rate=("accepted", "mean"),
        mean_confidence=("confidence", "mean"),
    )
    .assign(acceptance_rate=lambda table: (100 * table["acceptance_rate"]).round(1))
    .assign(mean_confidence=lambda table: table["mean_confidence"].round(3))
)

# Save validation-only diagnostics and the frozen policy settings.
plantdoc_val_scored.to_csv(
    OOD_DIR / "plantdoc_validation_threshold_results.csv", index=False
)
sugar_val_scored.to_csv(
    OOD_DIR / "sugarcane_validation_threshold_results.csv", index=False
)

print("\nSaved validation diagnostics under:", OOD_DIR)

PlantDoc validation threshold check:   0%|          | 0/88 [00:00<?, ?it/s]

Frozen confidence threshold: 0.578150
PlantDoc validation images: 349
PlantDoc validation rejection: 45.6%

PlantDoc validation rejection by class:


,images,rejected,rejection_rate,mean_confidence
plantdoc_class,,,,
Tomato mold leaf,13,11,84.6,0.493
grape leaf,9,7,77.8,0.548
Apple leaf,12,9,75.0,0.475
Tomato leaf,8,6,75.0,0.486
Peach leaf,15,10,66.7,0.565
Blueberry leaf,16,10,62.5,0.589
Bell_pepper leaf,8,5,62.5,0.636
Potato leaf early blight,16,10,62.5,0.576
Tomato leaf mosaic virus,7,4,57.1,0.637


Sugarcane validation threshold check:   0%|          | 0/84 [00:00<?, ?it/s]


Sugarcane validation accepted: 95.2%
Sugarcane validation acceptance by class:


,images,accepted,acceptance_rate,mean_confidence
label,,,,
Healthy,70,66,94.3,0.916
Mosaic,55,55,100.0,0.996
RedRot,74,73,98.6,0.945
Rust,66,57,86.4,0.882
Yellow,71,69,97.2,0.942



Saved validation diagnostics under: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data


In [28]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from PIL import Image, ImageOps
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)
OOD_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_ood_data"
)

# Load the same OE-best checkpoint used for the confidence diagnostics.
oe_checkpoint_path = OOD_DIR / "sugarcane_oe_best.pth"
oe_checkpoint = torch.load(oe_checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(oe_checkpoint["model_state_dict"], strict=True)
model.to(DEVICE).eval()

# Dataset returns normalized full image and four tiles, plus its path.
class SugarcaneEmbeddingDataset(Dataset):
    def __init__(self, dataframe, with_label=False):
        self.df = dataframe.reset_index(drop=True)
        self.with_label = with_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        path = str(row["image_path"])
        full, tiles = make_ood_views(path)

        if self.with_label:
            return full, tiles, CLASS_TO_IDX[row["label"]], path
        return full, tiles, path

def extract_embeddings(dataframe, description, with_label=False, batch_size=4):
    dataset = SugarcaneEmbeddingDataset(dataframe, with_label=with_label)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    embeddings, paths, labels, confidences = [], [], [], []
    model.eval()

    with torch.inference_mode():
        for batch in tqdm(loader, desc=description):
            if with_label:
                full, tiles, targets, batch_paths = batch
                labels.extend(targets.tolist())
            else:
                full, tiles, batch_paths = batch

            full = full.to(DEVICE, non_blocking=True)
            tiles = tiles.to(DEVICE, non_blocking=True)

            # Match the feature representation used by the trained classifier:
            # concatenate full-view and aggregated tile-view embeddings.
            full_features = model.encode_view(full)

            batch_size_now, tile_count, channels, height, width = tiles.shape
            flat_tiles = tiles.reshape(
                batch_size_now * tile_count, channels, height, width
            )
            tile_features = model.encode_view(flat_tiles).reshape(
                batch_size_now, tile_count, -1
            )

            tile_weights = torch.softmax(
                model.tile_score(tile_features).squeeze(-1), dim=1
            )
            weighted_tiles = (
                tile_features * tile_weights.unsqueeze(-1)
            ).sum(dim=1)
            strongest_tile = tile_features.max(dim=1).values
            combined = torch.cat(
                [full_features, weighted_tiles, strongest_tile], dim=1
            )

            # Use a normalized penultimate classifier representation for cosine kNN.
            representation = model.classifier[:4](combined)
            representation = F.normalize(representation.float(), dim=1)

            logits = model.classifier(representation)
            batch_conf = torch.softmax(logits, dim=1).max(dim=1).values

            embeddings.append(representation.cpu())
            confidences.extend(batch_conf.cpu().tolist())
            paths.extend(list(batch_paths))

    return (
        torch.cat(embeddings).numpy(),
        np.asarray(labels, dtype=np.int64) if with_label else None,
        paths,
        np.asarray(confidences, dtype=np.float32),
    )

# Build the supported reference bank from Sugarcane TRAIN only.
sugar_train_for_knn = pd.read_csv(EXPERIMENT_DIR / "train_manifest.csv")
reference_embeddings, _, reference_paths, _ = extract_embeddings(
    sugar_train_for_knn,
    "Sugarcane training reference embeddings",
    with_label=False,
)
reference_embeddings = F.normalize(
    torch.tensor(reference_embeddings, dtype=torch.float32), dim=1
)

# Reuse the previously saved validation score files to select only their paths/labels.
sugar_val_scored = pd.read_csv(
    OOD_DIR / "sugarcane_validation_threshold_results.csv"
)
plantdoc_val_scored = pd.read_csv(
    OOD_DIR / "plantdoc_validation_threshold_results.csv"
)

sugar_val_manifest = pd.read_csv(EXPERIMENT_DIR / "validation_manifest.csv")
plantdoc_val_manifest = pd.read_csv(OOD_DIR / "plantdoc_ood_validation_manifest.csv")

# Extract penultimate representations for validation only.
sugar_val_embeddings, sugar_val_labels, sugar_val_paths, sugar_val_conf = extract_embeddings(
    sugar_val_manifest,
    "Sugarcane validation embeddings",
    with_label=True,
)
plantdoc_val_embeddings, _, plantdoc_val_paths, plantdoc_val_conf = extract_embeddings(
    plantdoc_val_manifest,
    "PlantDoc validation embeddings",
    with_label=False,
)

def knn_similarity(query_embeddings, reference, k=10, query_batch=64):
    query = F.normalize(torch.tensor(query_embeddings, dtype=torch.float32), dim=1)
    scores = []
    for start in range(0, len(query), query_batch):
        sims = query[start:start + query_batch] @ reference.T
        topk = sims.topk(min(k, reference.shape[0]), dim=1).values
        scores.extend(topk.mean(dim=1).tolist())
    return np.asarray(scores, dtype=np.float32)

K = 10
sugar_val_knn = knn_similarity(sugar_val_embeddings, reference_embeddings, K)
plantdoc_val_knn = knn_similarity(plantdoc_val_embeddings, reference_embeddings, K)

print("Sugarcane validation embeddings:", sugar_val_embeddings.shape)
print("PlantDoc validation embeddings:", plantdoc_val_embeddings.shape)
print("Training reference bank:", tuple(reference_embeddings.shape))
print("KNN K:", K)
print("Sugarcane validation mean KNN similarity:", round(float(sugar_val_knn.mean()), 4))
print("PlantDoc validation mean KNN similarity:", round(float(plantdoc_val_knn.mean()), 4))

# Save these scores for policy calibration in the next cell.
combined_scores = pd.concat([
    pd.DataFrame({
        "image_path": sugar_val_paths,
        "group": "supported_sugarcane_validation",
        "label": sugar_val_manifest["label"].astype(str).tolist(),
        "confidence": sugar_val_conf,
        "knn_similarity": sugar_val_knn,
    }),
    pd.DataFrame({
        "image_path": plantdoc_val_paths,
        "group": "plantdoc_validation",
        "label": plantdoc_val_manifest.get(
            "plantdoc_class",
            pd.Series(["unknown"] * len(plantdoc_val_paths)),
        ).astype(str).tolist(),
        "confidence": plantdoc_val_conf,
        "knn_similarity": plantdoc_val_knn,
    }),
])

scores_path = OOD_DIR / "sugarcane_validation_confidence_knn_scores.csv"
combined_scores.to_csv(scores_path, index=False)
print("Saved validation confidence + KNN scores:", scores_path)

Sugarcane training reference embeddings:   0%|          | 0/392 [00:00<?, ?it/s]

RuntimeError: Given normalized_shape=[768], expected input with shape [*, 768], but got input of size[4, 512]

In [29]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)
OOD_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_ood_data"
)
BEST_OE_PATH = OOD_DIR / "sugarcane_oe_best.pth"

checkpoint = torch.load(BEST_OE_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.to(DEVICE).eval()

class SugarcaneEmbeddingDataset(Dataset):
    def __init__(self, dataframe, with_label=False):
        self.df = dataframe.reset_index(drop=True)
        self.with_label = with_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        full, tiles = make_ood_views(str(row["image_path"]))

        if self.with_label:
            return full, tiles, CLASS_TO_IDX[row["label"]], str(row["image_path"])
        return full, tiles, str(row["image_path"])

def extract_embeddings(dataframe, description, with_label=False, batch_size=4):
    dataset = SugarcaneEmbeddingDataset(dataframe, with_label=with_label)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    embeddings, paths, labels, confidences = [], [], [], []
    model.eval()

    with torch.inference_mode():
        for batch in tqdm(loader, desc=description):
            if with_label:
                full, tiles, targets, batch_paths = batch
                labels.extend(targets.tolist())
            else:
                full, tiles, batch_paths = batch

            full = full.to(DEVICE, non_blocking=True)
            tiles = tiles.to(DEVICE, non_blocking=True)

            # Regular forward pass is used for model confidence.
            logits = model(full, tiles)
            batch_confidence = torch.softmax(logits, dim=1).max(dim=1).values

            # Recreate the fused 768-dimensional classifier input.
            full_features = model.encode_view(full)
            batch_size_now, tile_count, channels, height, width = tiles.shape
            flat_tiles = tiles.reshape(
                batch_size_now * tile_count, channels, height, width
            )
            tile_features = model.encode_view(flat_tiles).reshape(
                batch_size_now, tile_count, -1
            )
            tile_weights = torch.softmax(
                model.tile_score(tile_features).squeeze(-1),
                dim=1,
            )
            weighted_tiles = (
                tile_features * tile_weights.unsqueeze(-1)
            ).sum(dim=1)
            strongest_tile = tile_features.max(dim=1).values
            combined = torch.cat(
                [full_features, weighted_tiles, strongest_tile],
                dim=1,
            )

            # Classifier modules: LayerNorm, Dropout, Linear(768->512), ReLU,
            # Dropout, Linear(512->5). Use the 512-d hidden feature before
            # the final classification layer for cosine kNN.
            hidden = model.classifier[0](combined)
            hidden = model.classifier[1](hidden)
            hidden = model.classifier[2](hidden)
            hidden = model.classifier[3](hidden)
            embedding = F.normalize(hidden.float(), dim=1)

            embeddings.append(embedding.cpu())
            confidences.extend(batch_confidence.cpu().tolist())
            paths.extend(list(batch_paths))

    embeddings = torch.cat(embeddings).numpy()
    labels = np.asarray(labels, dtype=np.int64) if with_label else None
    return embeddings, labels, paths, np.asarray(confidences, dtype=np.float32)

def knn_similarity(query_embeddings, reference_embeddings, k=10, batch_size=64):
    query = F.normalize(
        torch.tensor(query_embeddings, dtype=torch.float32), dim=1
    )
    reference = F.normalize(
        torch.tensor(reference_embeddings, dtype=torch.float32), dim=1
    )

    scores = []
    for start in range(0, len(query), batch_size):
        similarities = query[start:start + batch_size] @ reference.T
        top_k = similarities.topk(min(k, reference.shape[0]), dim=1).values
        scores.extend(top_k.mean(dim=1).tolist())

    return np.asarray(scores, dtype=np.float32)

# Reference bank uses Sugarcane training images only.
sugar_train_df = pd.read_csv(EXPERIMENT_DIR / "train_manifest.csv")
reference_embeddings, _, reference_paths, _ = extract_embeddings(
    sugar_train_df,
    "Sugarcane training reference embeddings",
    with_label=False,
)

# Calibration scores use Sugarcane validation and PlantDoc calibration only.
sugar_val_df = pd.read_csv(EXPERIMENT_DIR / "validation_manifest.csv")
plantdoc_val_df = pd.read_csv(OOD_DIR / "plantdoc_ood_validation_manifest.csv")

sugar_embeddings, sugar_labels, sugar_paths, sugar_conf = extract_embeddings(
    sugar_val_df,
    "Sugarcane validation embeddings",
    with_label=True,
)
plantdoc_embeddings, _, plantdoc_paths, plantdoc_conf = extract_embeddings(
    plantdoc_val_df,
    "PlantDoc validation embeddings",
    with_label=False,
)

K = 10
sugar_knn = knn_similarity(sugar_embeddings, reference_embeddings, k=K)
plantdoc_knn = knn_similarity(plantdoc_embeddings, reference_embeddings, k=K)

print("Reference embedding shape:", reference_embeddings.shape)
print("Sugarcane validation embedding shape:", sugar_embeddings.shape)
print("PlantDoc validation embedding shape:", plantdoc_embeddings.shape)
print("Sugarcane validation mean KNN:", round(float(sugar_knn.mean()), 4))
print("PlantDoc validation mean KNN:", round(float(plantdoc_knn.mean()), 4))

scores_df = pd.concat([
    pd.DataFrame({
        "image_path": sugar_paths,
        "group": "supported_sugarcane_validation",
        "label": sugar_val_df["label"].astype(str).tolist(),
        "confidence": sugar_conf,
        "knn_similarity": sugar_knn,
    }),
    pd.DataFrame({
        "image_path": plantdoc_paths,
        "group": "plantdoc_validation",
        "label": plantdoc_val_df["plantdoc_class"].astype(str).tolist(),
        "confidence": plantdoc_conf,
        "knn_similarity": plantdoc_knn,
    }),
], ignore_index=True)

# Also score the separate PlantDoc calibration split for policy fitting.
plantdoc_cal_df = pd.read_csv(OOD_DIR / "plantdoc_ood_calibration_manifest.csv")
cal_embeddings, _, cal_paths, cal_conf = extract_embeddings(
    plantdoc_cal_df,
    "PlantDoc calibration embeddings",
    with_label=False,
)
cal_knn = knn_similarity(cal_embeddings, reference_embeddings, k=K)

calibration_df = pd.DataFrame({
    "image_path": cal_paths,
    "group": "plantdoc_calibration",
    "label": plantdoc_cal_df["plantdoc_class"].astype(str).tolist(),
    "confidence": cal_conf,
    "knn_similarity": cal_knn,
})

scores_df = pd.concat([scores_df, calibration_df], ignore_index=True)

scores_path = OOD_DIR / "sugarcane_validation_confidence_knn_scores.csv"
scores_df.to_csv(scores_path, index=False)

# Save the training-only reference bank for reproducibility.
reference_path = OOD_DIR / "sugarcane_training_reference_embeddings.pt"
torch.save(
    {
        "embeddings": torch.tensor(reference_embeddings, dtype=torch.float32),
        "image_paths": reference_paths,
        "checkpoint": str(BEST_OE_PATH),
        "k": K,
    },
    reference_path,
)

print("Saved validation/calibration scores:", scores_path)
print("Saved training reference bank:", reference_path)

Sugarcane training reference embeddings:   0%|          | 0/392 [00:00<?, ?it/s]

Sugarcane validation embeddings:   0%|          | 0/84 [00:00<?, ?it/s]

PlantDoc validation embeddings:   0%|          | 0/88 [00:00<?, ?it/s]

Reference embedding shape: (1568, 512)
Sugarcane validation embedding shape: (336, 512)
PlantDoc validation embedding shape: (349, 512)
Sugarcane validation mean KNN: 0.9445
PlantDoc validation mean KNN: 0.6907


PlantDoc calibration embeddings:   0%|          | 0/88 [00:00<?, ?it/s]

Saved validation/calibration scores: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/sugarcane_validation_confidence_knn_scores.csv
Saved training reference bank: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/sugarcane_training_reference_embeddings.pt


In [32]:
import pandas as pd
from pathlib import Path

OOD_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_ood_data"
)
SCORES_PATH = OOD_DIR / "sugarcane_validation_confidence_knn_scores.csv"

scores = pd.read_csv(SCORES_PATH)
supported = scores[scores["group"] == "supported_sugarcane_validation"].copy()
ood_cal = scores[scores["group"] == "plantdoc_calibration"].copy()

if supported.empty or ood_cal.empty:
    raise ValueError("Missing Sugarcane validation or PlantDoc calibration scores.")

CONFIDENCE_THRESHOLD = 0.578150
TARGETS = [0.99, 0.98, 0.95, 0.92, 0.90, 0.85]

candidate_knn_thresholds = sorted(
    set(supported["knn_similarity"].tolist())
    | set(ood_cal["knn_similarity"].tolist())
)

rows = []
for knn_threshold in candidate_knn_thresholds:
    supported_accepted = (
        (supported["confidence"] >= CONFIDENCE_THRESHOLD)
        & (supported["knn_similarity"] >= knn_threshold)
    )
    ood_rejected = (
        (ood_cal["confidence"] < CONFIDENCE_THRESHOLD)
        | (ood_cal["knn_similarity"] < knn_threshold)
    )

    by_class = (
        supported.assign(accepted=supported_accepted)
        .groupby("label")["accepted"]
        .mean()
        .to_dict()
    )

    rows.append({
        "knn_threshold": float(knn_threshold),
        "supported_acceptance": float(supported_accepted.mean()),
        "minimum_class_acceptance": float(min(by_class.values())),
        "plantdoc_calibration_rejection": float(ood_rejected.mean()),
        **{f"class_accept_{name}": rate for name, rate in by_class.items()},
    })

sweep = pd.DataFrame(rows)

print(f"Frozen confidence threshold: {CONFIDENCE_THRESHOLD:.6f}")
print("\nBest available combined-policy tradeoff at each overall acceptance target:")

best_rows = []
for target in TARGETS:
    eligible = sweep[sweep["supported_acceptance"] >= target]
    if eligible.empty:
        continue

    # Maximize PlantDoc rejection while meeting the overall support target.
    chosen = eligible.sort_values(
        ["plantdoc_calibration_rejection", "supported_acceptance"],
        ascending=[False, False],
    ).iloc[0]

    row = chosen.to_dict()
    row["target_supported_acceptance"] = target
    best_rows.append(row)

tradeoffs = pd.DataFrame(best_rows)
display(
    tradeoffs[
        [
            "target_supported_acceptance",
            "knn_threshold",
            "supported_acceptance",
            "minimum_class_acceptance",
            "plantdoc_calibration_rejection",
        ]
        + [column for column in tradeoffs.columns if column.startswith("class_accept_")]
    ].round(4)
)

sweep_path = OOD_DIR / "sugarcane_oe_knn_threshold_sweep.csv"
tradeoffs_path = OOD_DIR / "sugarcane_oe_knn_tradeoff_summary.csv"

sweep.to_csv(sweep_path, index=False)
tradeoffs.to_csv(tradeoffs_path, index=False)

print("\nSaved full threshold sweep:", sweep_path)
print("Saved tradeoff summary:", tradeoffs_path)

Frozen confidence threshold: 0.578150

Best available combined-policy tradeoff at each overall acceptance target:


,target_supported_acceptance,knn_threshold,supported_acceptance,minimum_class_acceptance,plantdoc_calibration_rejection,class_accept_Healthy,class_accept_Mosaic,class_accept_RedRot,class_accept_Rust,class_accept_Yellow
0,0.95,0.8175,0.9554,0.8636,0.8968,0.9857,1.0,0.9730,0.8636,0.9577
1,0.92,0.8695,0.9226,0.8030,0.9570,0.9571,1.0,0.9324,0.8030,0.9296
2,0.90,0.8793,0.9107,0.8030,0.9628,0.9429,1.0,0.9189,0.8030,0.9014
3,0.85,0.9014,0.8690,0.7273,0.9799,0.9286,1.0,0.8784,0.7273,0.8310



Saved full threshold sweep: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/sugarcane_oe_knn_threshold_sweep.csv
Saved tradeoff summary: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/sugarcane_oe_knn_tradeoff_summary.csv


In [33]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)
OOD_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_ood_data"
)

CONFIDENCE_THRESHOLD = 0.578150
KNN_THRESHOLD = 0.8175
K = 10

# Load the saved training-only Sugarcane reference bank.
bank_path = OOD_DIR / "sugarcane_training_reference_embeddings.pt"
bank = torch.load(bank_path, map_location="cpu", weights_only=False)
reference_embeddings = F.normalize(bank["embeddings"].float(), dim=1)

# Load best OE checkpoint.
checkpoint_path = OOD_DIR / "sugarcane_oe_best.pth"
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.to(DEVICE).eval()

class PolicyDataset(Dataset):
    def __init__(self, dataframe, supported):
        self.df = dataframe.reset_index(drop=True)
        self.supported = supported

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        full, tiles = make_ood_views(str(row["image_path"]))
        if self.supported:
            return full, tiles, str(row["image_path"]), str(row["label"])
        return full, tiles, str(row["image_path"]), str(row.get("plantdoc_class", "unknown"))

def extract_policy_scores(dataframe, description, supported):
    loader = DataLoader(
        PolicyDataset(dataframe, supported),
        batch_size=4,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    rows = []

    with torch.inference_mode():
        for full, tiles, paths, labels in tqdm(loader, desc=description):
            full = full.to(DEVICE, non_blocking=True)
            tiles = tiles.to(DEVICE, non_blocking=True)

            logits = model(full, tiles)
            confidence = torch.softmax(logits, dim=1).max(dim=1).values

            full_features = model.encode_view(full)
            batch_size, tile_count, channels, height, width = tiles.shape
            flat_tiles = tiles.reshape(batch_size * tile_count, channels, height, width)
            tile_features = model.encode_view(flat_tiles).reshape(batch_size, tile_count, -1)
            tile_weights = torch.softmax(
                model.tile_score(tile_features).squeeze(-1), dim=1
            )
            weighted_tiles = (tile_features * tile_weights.unsqueeze(-1)).sum(dim=1)
            strongest_tile = tile_features.max(dim=1).values
            combined = torch.cat(
                [full_features, weighted_tiles, strongest_tile], dim=1
            )

            hidden = model.classifier[0](combined)
            hidden = model.classifier[1](hidden)
            hidden = model.classifier[2](hidden)
            hidden = model.classifier[3](hidden)
            embeddings = F.normalize(hidden.float(), dim=1).cpu()

            similarities = embeddings @ reference_embeddings.T
            knn_scores = similarities.topk(
                min(K, reference_embeddings.shape[0]), dim=1
            ).values.mean(dim=1)

            for i in range(len(paths)):
                rows.append({
                    "image_path": paths[i],
                    "label": labels[i],
                    "confidence": float(confidence[i].cpu()),
                    "knn_similarity": float(knn_scores[i]),
                })

    result = pd.DataFrame(rows)
    result["accepted"] = (
        (result["confidence"] >= CONFIDENCE_THRESHOLD)
        & (result["knn_similarity"] >= KNN_THRESHOLD)
    )
    result["rejected_as_ood"] = ~result["accepted"]
    return result

# These are validation partitions only.
sugar_val_df = pd.read_csv(EXPERIMENT_DIR / "validation_manifest.csv")
plantdoc_val_df = pd.read_csv(OOD_DIR / "plantdoc_ood_validation_manifest.csv")

sugar_result = extract_policy_scores(
    sugar_val_df, "Sugarcane validation policy check", supported=True
)
plantdoc_result = extract_policy_scores(
    plantdoc_val_df, "PlantDoc validation policy check", supported=False
)

print(f"Frozen confidence threshold: {CONFIDENCE_THRESHOLD:.6f}")
print(f"Frozen kNN threshold: {KNN_THRESHOLD:.4f}; K={K}")
print(f"\nSugarcane validation acceptance: {sugar_result['accepted'].mean():.1%}")
print(f"PlantDoc validation rejection: {plantdoc_result['rejected_as_ood'].mean():.1%}")

print("\nSugarcane acceptance by class:")
display(
    sugar_result.groupby("label")
    .agg(
        images=("image_path", "size"),
        accepted=("accepted", "sum"),
        acceptance_rate=("accepted", "mean"),
    )
    .assign(acceptance_rate=lambda x: (100 * x["acceptance_rate"]).round(1))
)

print("\nPlantDoc rejection by class:")
display(
    plantdoc_result.groupby("label")
    .agg(
        images=("image_path", "size"),
        rejected=("rejected_as_ood", "sum"),
        rejection_rate=("rejected_as_ood", "mean"),
    )
    .assign(rejection_rate=lambda x: (100 * x["rejection_rate"]).round(1))
    .sort_values("rejection_rate", ascending=False)
)

sugar_path = OOD_DIR / "sugarcane_validation_combined_policy_results.csv"
plantdoc_path = OOD_DIR / "plantdoc_validation_combined_policy_results.csv"
sugar_result.to_csv(sugar_path, index=False)
plantdoc_result.to_csv(plantdoc_path, index=False)

config = {
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "knn_threshold": KNN_THRESHOLD,
    "k": K,
    "calibration_supported_acceptance": 0.9554,
    "calibration_plantdoc_rejection": 0.8968,
    "validation_sugarcane_acceptance": float(sugar_result["accepted"].mean()),
    "validation_plantdoc_rejection": float(plantdoc_result["rejected_as_ood"].mean()),
    "validation_sugarcane_acceptance_by_class": (
        sugar_result.groupby("label")["accepted"].mean().to_dict()
    ),
    "validation_plantdoc_rejection_by_class": (
        plantdoc_result.groupby("label")["rejected_as_ood"].mean().to_dict()
    ),
    "selected_using": ["Sugarcane validation", "PlantDoc calibration"],
    "held_out_test_used": False,
}
config_path = OOD_DIR / "sugarcane_oe_knn_candidate_policy.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("\nSaved results:")
print(sugar_path)
print(plantdoc_path)
print(config_path)

Sugarcane validation policy check:   0%|          | 0/84 [00:00<?, ?it/s]

PlantDoc validation policy check:   0%|          | 0/88 [00:00<?, ?it/s]

Frozen confidence threshold: 0.578150
Frozen kNN threshold: 0.8175; K=10

Sugarcane validation acceptance: 95.2%
PlantDoc validation rejection: 87.7%

Sugarcane acceptance by class:


,images,accepted,acceptance_rate
label,,,
Healthy,70,69,98.6
Mosaic,55,55,100.0
RedRot,74,72,97.3
Rust,66,56,84.8
Yellow,71,68,95.8



PlantDoc rejection by class:


,images,rejected,rejection_rate
label,,,
Apple leaf,12,12,100.0
Blueberry leaf,16,16,100.0
Bell_pepper leaf,8,8,100.0
Peach leaf,15,15,100.0
grape leaf black rot,8,8,100.0
grape leaf,9,9,100.0
Tomato mold leaf,13,13,100.0
Tomato leaf,8,8,100.0
Tomato Septoria leaf spot,21,21,100.0



Saved results:
/content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/sugarcane_validation_combined_policy_results.csv
/content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/plantdoc_validation_combined_policy_results.csv
/content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/sugarcane_oe_knn_candidate_policy.json


In [34]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from tqdm.auto import tqdm

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)
OOD_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_ood_data"
)

BASELINE_PATH = OOD_DIR / "sugarcane_baseline_before_ood.pth"
OE_PATH = OOD_DIR / "sugarcane_oe_best.pth"
REFERENCE_PATH = OOD_DIR / "sugarcane_training_reference_embeddings.pt"
SUGARCANE_TEST_CSV = EXPERIMENT_DIR / "internal_test_manifest.csv"
PLANTDOC_TEST_CSV = OOD_DIR / "plantdoc_ood_test_manifest.csv"

CONFIDENCE_THRESHOLD = 0.578150
KNN_THRESHOLD = 0.8175
K = 10
BATCH_SIZE = 4

for path in [BASELINE_PATH, OE_PATH, REFERENCE_PATH, SUGARCANE_TEST_CSV, PLANTDOC_TEST_CSV]:
    if not path.exists():
        raise FileNotFoundError(f"Required file missing: {path}")

reference_bank = torch.load(REFERENCE_PATH, map_location="cpu", weights_only=False)
reference_embeddings = F.normalize(reference_bank["embeddings"].float(), dim=1)

class TestImageDataset(Dataset):
    def __init__(self, dataframe, supported):
        self.df = dataframe.reset_index(drop=True)
        self.supported = supported

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        full, tiles = make_ood_views(str(row["image_path"]))
        if self.supported:
            return full, tiles, int(CLASS_TO_IDX[row["label"]]), str(row["image_path"])
        return full, tiles, str(row.get("plantdoc_class", "unknown")), str(row["image_path"])

def hidden_embeddings_from_views(full, tiles):
    full_features = model.encode_view(full)

    batch, tile_count, channels, height, width = tiles.shape
    flat_tiles = tiles.reshape(batch * tile_count, channels, height, width)
    tile_features = model.encode_view(flat_tiles).reshape(batch, tile_count, -1)

    weights = torch.softmax(model.tile_score(tile_features).squeeze(-1), dim=1)
    weighted_tiles = (tile_features * weights.unsqueeze(-1)).sum(dim=1)
    strongest_tile = tile_features.max(dim=1).values
    combined = torch.cat([full_features, weighted_tiles, strongest_tile], dim=1)

    hidden = model.classifier[0](combined)
    hidden = model.classifier[1](hidden)
    hidden = model.classifier[2](hidden)
    hidden = model.classifier[3](hidden)
    return F.normalize(hidden.float(), dim=1)

def evaluate_supported_checkpoint(checkpoint_path, description, apply_policy=False):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.to(DEVICE).eval()

    test_df = pd.read_csv(SUGARCANE_TEST_CSV)
    loader = DataLoader(
        TestImageDataset(test_df, supported=True),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    y_true, y_pred, confidences, knn_scores = [], [], [], []
    with torch.inference_mode():
        for full, tiles, targets, _paths in tqdm(loader, desc=description):
            full = full.to(DEVICE, non_blocking=True)
            tiles = tiles.to(DEVICE, non_blocking=True)

            probabilities = torch.softmax(model(full, tiles), dim=1)
            confidence, prediction = probabilities.max(dim=1)
            y_true.extend(targets.tolist())
            y_pred.extend(prediction.cpu().tolist())
            confidences.extend(confidence.cpu().tolist())

            if apply_policy:
                embeddings = hidden_embeddings_from_views(full, tiles).cpu()
                similarities = embeddings @ reference_embeddings.T
                knn = similarities.topk(
                    min(K, reference_embeddings.shape[0]), dim=1
                ).values.mean(dim=1)
                knn_scores.extend(knn.tolist())

    metrics = {
        "images": len(y_true),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    }

    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(CLASS_NAMES))),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))

    print(f"\n{description}")
    print("Checkpoint epoch:", checkpoint.get("epoch", "not recorded"))
    print("Metrics:", metrics)
    display(pd.DataFrame(report).T.round(4))
    display(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES))

    if apply_policy:
        accepted = (
            (np.asarray(confidences) >= CONFIDENCE_THRESHOLD)
            & (np.asarray(knn_scores) >= KNN_THRESHOLD)
        )
        metrics["policy_accepted"] = int(accepted.sum())
        metrics["policy_acceptance"] = float(accepted.mean())
        metrics["accuracy_among_accepted"] = (
            float(accuracy_score(np.asarray(y_true)[accepted], np.asarray(y_pred)[accepted]))
            if accepted.any() else None
        )
        metrics["acceptance_by_class"] = {}
        for class_index, class_name in enumerate(CLASS_NAMES):
            class_mask = np.asarray(y_true) == class_index
            metrics["acceptance_by_class"][class_name] = float(accepted[class_mask].mean())

        print(
            f"Frozen policy accepted {accepted.sum()}/{len(accepted)} "
            f"({accepted.mean():.1%}) Sugarcane test images."
        )
        print("Accuracy among accepted:", metrics["accuracy_among_accepted"])
        print("Acceptance by true class:", metrics["acceptance_by_class"])

    return metrics, report, cm

# First assess the unchanged supervised model as a reference.
baseline_metrics, baseline_report, baseline_cm = evaluate_supported_checkpoint(
    BASELINE_PATH,
    "Supervised baseline — Sugarcane internal test",
    apply_policy=False,
)

# Then assess the OE checkpoint and frozen reject policy.
oe_metrics, oe_report, oe_cm = evaluate_supported_checkpoint(
    OE_PATH,
    "OE model — Sugarcane internal test",
    apply_policy=True,
)

# Evaluate the held-out PlantDoc OOD test set once with the same frozen policy.
plantdoc_test_df = pd.read_csv(PLANTDOC_TEST_CSV)
plantdoc_loader = DataLoader(
    TestImageDataset(plantdoc_test_df, supported=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

# Ensure the OE model is loaded for OOD test scoring.
oe_checkpoint = torch.load(OE_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(oe_checkpoint["model_state_dict"], strict=True)
model.to(DEVICE).eval()

plantdoc_test_rows = []
with torch.inference_mode():
    for full, tiles, labels, paths in tqdm(
        plantdoc_loader, desc="Held-out PlantDoc OOD test"
    ):
        full = full.to(DEVICE, non_blocking=True)
        tiles = tiles.to(DEVICE, non_blocking=True)

        probabilities = torch.softmax(model(full, tiles), dim=1)
        confidence = probabilities.max(dim=1).values
        embeddings = hidden_embeddings_from_views(full, tiles).cpu()
        similarities = embeddings @ reference_embeddings.T
        knn = similarities.topk(min(K, reference_embeddings.shape[0]), dim=1).values.mean(dim=1)

        for i in range(len(paths)):
            accepted = (
                float(confidence[i].cpu()) >= CONFIDENCE_THRESHOLD
                and float(knn[i]) >= KNN_THRESHOLD
            )
            plantdoc_test_rows.append({
                "image_path": paths[i],
                "plantdoc_class": labels[i],
                "confidence": float(confidence[i].cpu()),
                "knn_similarity": float(knn[i]),
                "accepted_as_sugarcane": accepted,
                "rejected_as_ood": not accepted,
            })

plantdoc_test_result = pd.DataFrame(plantdoc_test_rows)
plantdoc_rejection = float(plantdoc_test_result["rejected_as_ood"].mean())

print("\nHeld-out PlantDoc OOD test")
print("Images:", len(plantdoc_test_result))
print(f"Rejected: {plantdoc_test_result['rejected_as_ood'].sum()}/{len(plantdoc_test_result)} ({plantdoc_rejection:.1%})")
print("\nRejection by PlantDoc class:")
display(
    plantdoc_test_result.groupby("plantdoc_class")
    .agg(
        images=("image_path", "size"),
        rejected=("rejected_as_ood", "sum"),
        rejection_rate=("rejected_as_ood", "mean"),
    )
    .assign(rejection_rate=lambda x: (100 * x["rejection_rate"]).round(1))
    .sort_values("rejection_rate", ascending=False)
)

plantdoc_test_result.to_csv(
    OOD_DIR / "plantdoc_ood_test_frozen_policy_results.csv",
    index=False,
)

final_results = {
    "frozen_policy": {
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "knn_threshold": KNN_THRESHOLD,
        "k": K,
    },
    "baseline_sugarcane_test": baseline_metrics,
    "oe_sugarcane_test": oe_metrics,
    "plantdoc_ood_test_images": len(plantdoc_test_result),
    "plantdoc_ood_test_rejection": plantdoc_rejection,
}
with open(OOD_DIR / "sugarcane_final_ood_test_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

print("\nSaved held-out PlantDoc results and summary under:", OOD_DIR)

Supervised baseline — Sugarcane internal test:   0%|          | 0/84 [00:00<?, ?it/s]


Supervised baseline — Sugarcane internal test
Checkpoint epoch: 11
Metrics: {'images': 336, 'accuracy': 0.9553571428571429, 'balanced_accuracy': 0.9570544926791099, 'macro_f1': 0.9561452286694598}


,precision,recall,f1-score,support
Healthy,0.9067,0.9855,0.9444,69.0000
Mosaic,0.9649,0.9821,0.9735,56.0000
RedRot,0.9718,0.9324,0.9517,74.0000
Rust,0.9412,0.9697,0.9552,66.0000
Yellow,1.0000,0.9155,0.9559,71.0000
accuracy,0.9554,0.9554,0.9554,0.9554
macro avg,0.9569,0.9571,0.9561,336.0000
weighted avg,0.9572,0.9554,0.9554,336.0000


,Healthy,Mosaic,RedRot,Rust,Yellow
Healthy,68,1,0,0,0
Mosaic,0,55,0,1,0
RedRot,4,0,69,1,0
Rust,1,0,1,64,0
Yellow,2,1,1,2,65


OE model — Sugarcane internal test:   0%|          | 0/84 [00:00<?, ?it/s]

...

Frozen policy accepted 325/336 (96.7%) Sugarcane test images.
Accuracy among accepted: 0.9384615384615385
Acceptance by true class: {'Healthy': 0.9710144927536232, 'Mosaic': 1.0, 'RedRot': 0.9864864864864865, 'Rust': 0.9545454545454546, 'Yellow': 0.9295774647887324}


Held-out PlantDoc OOD test:   0%|          | 0/57 [00:00<?, ?it/s]


Held-out PlantDoc OOD test
Images: 228
Rejected: 204/228 (89.5%)

Rejection by PlantDoc class:


,images,rejected,rejection_rate
plantdoc_class,,,
Apple Scab Leaf,10,10,100.0
Apple leaf,9,9,100.0
Apple rust leaf,10,10,100.0
grape leaf,12,12,100.0
grape leaf black rot,8,8,100.0
Tomato Early blight leaf,9,9,100.0
Tomato Septoria leaf spot,11,11,100.0
Strawberry leaf,8,8,100.0
Soyabean leaf,8,8,100.0



Saved held-out PlantDoc results and summary under: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data


In [35]:
import json
from pathlib import Path

OOD_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_ood_data"
)

candidate_path = OOD_DIR / "sugarcane_oe_knn_candidate_policy.json"
test_results_path = OOD_DIR / "sugarcane_final_ood_test_results.json"
final_path = OOD_DIR / "sugarcane_reliability_config.json"

with open(candidate_path) as f:
    candidate = json.load(f)

with open(test_results_path) as f:
    test_results = json.load(f)

final_config = {
    "crop": "sugarcane",
    "model_version": "1.0.0",
    "supported_classes": ["Healthy", "Mosaic", "RedRot", "Rust", "Yellow"],
    "decision_rule": {
        "accept_if": (
            "maximum softmax confidence >= confidence_threshold AND "
            "mean cosine similarity to the k nearest Sugarcane training "
            "embeddings >= knn_threshold"
        ),
        "confidence_threshold": candidate["confidence_threshold"],
        "knn_threshold": candidate["knn_threshold"],
        "knn_k": candidate["k"],
    },
    "reference_bank": "sugarcane_training_reference_embeddings.pt",
    "checkpoint": "sugarcane_oe_best.pth",
    "calibration": {
        "sources": candidate["selected_using"],
        "sugarcane_validation_acceptance": candidate[
            "validation_sugarcane_acceptance"
        ],
        "plantdoc_validation_rejection": candidate[
            "validation_plantdoc_rejection"
        ],
        "sugarcane_acceptance_by_class": candidate[
            "validation_sugarcane_acceptance_by_class"
        ],
        "plantdoc_rejection_by_class": candidate[
            "validation_plantdoc_rejection_by_class"
        ],
    },
    "held_out_test_results": {
        "baseline_sugarcane_test": test_results["baseline_sugarcane_test"],
        "oe_sugarcane_test": test_results["oe_sugarcane_test"],
        "plantdoc_test_images": test_results["plantdoc_ood_test_images"],
        "plantdoc_test_rejection": test_results["plantdoc_ood_test_rejection"],
    },
    "limitations": [
        "OOD rejection was evaluated on PlantDoc other-crop images.",
        "Performance on unsupported Sugarcane diseases has not been measured.",
        "The model rejects unsupported inputs; it does not identify their disease.",
    ],
}

with open(final_path, "w") as f:
    json.dump(final_config, f, indent=2)

print("Frozen reliability config saved:", final_path)
print("Confidence threshold:", final_config["decision_rule"]["confidence_threshold"])
print("kNN threshold:", final_config["decision_rule"]["knn_threshold"])

Frozen reliability config saved: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/sugarcane_reliability_config.json
Confidence threshold: 0.57815
kNN threshold: 0.8175


In [36]:
import json
import shutil
import sys
from pathlib import Path

import torch

OOD_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_ood_data"
)
EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_model_experiment"
)
PACKAGE_DIR = OOD_DIR / "hf_package" / "sugarcane" / "v1"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_CHECKPOINT = OOD_DIR / "sugarcane_oe_best.pth"
SOURCE_REFERENCE = OOD_DIR / "sugarcane_training_reference_embeddings.pt"
SOURCE_RELIABILITY = OOD_DIR / "sugarcane_reliability_config.json"

for path in [SOURCE_CHECKPOINT, SOURCE_REFERENCE, SOURCE_RELIABILITY]:
    if not path.exists():
        raise FileNotFoundError(f"Required package source is missing: {path}")

# This is the exact architecture used by the Sugarcane training cell.
model_source = r'''
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50


class SpatialSelfAttention(nn.Module):
    def __init__(self, channels=256, heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.attention = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=heads,
            batch_first=True,
        )
        self.gate = nn.Conv2d(channels * 2, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        normalized = self.norm(tokens)
        attended, _ = self.attention(
            normalized, normalized, normalized, need_weights=False
        )
        attended = attended.transpose(1, 2).reshape(
            batch, channels, height, width
        )
        gate = torch.sigmoid(self.gate(torch.cat([x, attended], dim=1)))
        return x + gate * attended


class SugarcaneResNet50Attention(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        backbone = resnet50(weights=None)

        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.project2 = nn.Conv2d(512, 256, kernel_size=1)
        self.project3 = nn.Conv2d(1024, 256, kernel_size=1)
        self.project4 = nn.Conv2d(2048, 256, kernel_size=1)

        self.fusion = nn.Sequential(
            nn.Conv2d(768, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.self_attention = SpatialSelfAttention(channels=256, heads=8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.tile_score = nn.Linear(256, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.35),
            nn.Linear(768, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(512, num_classes),
        )

    def encode_view(self, images):
        x = self.stem(images)
        x = self.layer1(x)
        fine = self.layer2(x)
        middle = self.layer3(fine)
        deep = self.layer4(middle)

        output_size = fine.shape[-2:]
        fine = self.project2(fine)
        middle = F.interpolate(
            self.project3(middle),
            size=output_size,
            mode="bilinear",
            align_corners=False,
        )
        deep = F.interpolate(
            self.project4(deep),
            size=output_size,
            mode="bilinear",
            align_corners=False,
        )

        fused = self.fusion(torch.cat([fine, middle, deep], dim=1))
        attended = self.self_attention(fused)
        return self.pool(attended).flatten(1)

    def forward(self, full_images, tile_images):
        full_features = self.encode_view(full_images)

        batch, tile_count, channels, height, width = tile_images.shape
        flat_tiles = tile_images.reshape(
            batch * tile_count, channels, height, width
        )
        tile_features = self.encode_view(flat_tiles).reshape(
            batch, tile_count, -1
        )

        tile_weights = torch.softmax(
            self.tile_score(tile_features).squeeze(-1), dim=1
        )
        weighted_tiles = (
            tile_features * tile_weights.unsqueeze(-1)
        ).sum(dim=1)
        strongest_tile = tile_features.max(dim=1).values

        combined = torch.cat(
            [full_features, weighted_tiles, strongest_tile], dim=1
        )
        return self.classifier(combined)

    def extract_embedding(self, full_images, tile_images):
        full_features = self.encode_view(full_images)

        batch, tile_count, channels, height, width = tile_images.shape
        flat_tiles = tile_images.reshape(
            batch * tile_count, channels, height, width
        )
        tile_features = self.encode_view(flat_tiles).reshape(
            batch, tile_count, -1
        )

        tile_weights = torch.softmax(
            self.tile_score(tile_features).squeeze(-1), dim=1
        )
        weighted_tiles = (
            tile_features * tile_weights.unsqueeze(-1)
        ).sum(dim=1)
        strongest_tile = tile_features.max(dim=1).values
        combined = torch.cat(
            [full_features, weighted_tiles, strongest_tile], dim=1
        )

        hidden = self.classifier[0](combined)
        hidden = self.classifier[1](hidden)
        hidden = self.classifier[2](hidden)
        hidden = self.classifier[3](hidden)
        return F.normalize(hidden.float(), dim=1)
'''

inference_source = r'''
import json
from pathlib import Path

import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from torchvision.transforms import functional as TF

from model import SugarcaneResNet50Attention


class SugarcaneDiseasePredictor:
    def __init__(self, model_dir, device=None):
        self.model_dir = Path(model_dir)
        self.device = torch.device(
            device or ("cuda" if torch.cuda.is_available() else "cpu")
        )

        self.classes = json.loads(
            (self.model_dir / "classes.json").read_text()
        )["classes"]
        self.config = json.loads(
            (self.model_dir / "sugarcane_reliability_config.json").read_text()
        )

        self.model = SugarcaneResNet50Attention(num_classes=len(self.classes))
        checkpoint = torch.load(
            self.model_dir / "sugarcane_oe_best.pth",
            map_location="cpu",
            weights_only=False,
        )
        self.model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        self.model.to(self.device).eval()

        bank = torch.load(
            self.model_dir / "sugarcane_knn_reference_bank.pt",
            map_location="cpu",
            weights_only=False,
        )
        self.reference_embeddings = F.normalize(
            bank["embeddings"].float(), dim=1
        ).to(self.device)

        policy = self.config["decision_rule"]
        self.confidence_threshold = float(policy["confidence_threshold"])
        self.knn_threshold = float(policy["knn_threshold"])
        self.k = int(policy["knn_k"])

        self.mean = (0.485, 0.456, 0.406)
        self.std = (0.229, 0.224, 0.225)

    def _views(self, image_path):
        with Image.open(image_path) as source:
            image = ImageOps.exif_transpose(source).convert("RGB")
            image = ImageOps.pad(
                image,
                (448, 448),
                method=Image.Resampling.LANCZOS,
                color=(127, 127, 127),
            )

        full = TF.normalize(TF.to_tensor(image), self.mean, self.std)
        starts = [(0, 0), (160, 0), (0, 160), (160, 160)]
        tiles = []
        for left, top in starts:
            tile = image.crop((left, top, left + 288, top + 288))
            tiles.append(TF.normalize(TF.to_tensor(tile), self.mean, self.std))

        return full.unsqueeze(0).to(self.device), torch.stack(tiles).unsqueeze(0).to(self.device)

    @torch.inference_mode()
    def predict(self, image_path):
        full, tiles = self._views(image_path)
        probabilities = torch.softmax(self.model(full, tiles), dim=1)[0]
        confidence, prediction = probabilities.max(dim=0)

        embedding = self.model.extract_embedding(full, tiles)
        similarities = embedding @ self.reference_embeddings.T
        knn_score = similarities.topk(
            min(self.k, self.reference_embeddings.shape[0]), dim=1
        ).values.mean().item()

        accepted = (
            confidence.item() >= self.confidence_threshold
            and knn_score >= self.knn_threshold
        )
        top_indices = probabilities.topk(min(3, len(self.classes)).indices

        return {
            "crop": "sugarcane",
            "model_version": "1.0.0",
            "decision": "PREDICTION" if accepted else "REJECTED",
            "predicted_class": self.classes[prediction.item()] if accepted else None,
            "confidence": float(confidence.item()),
            "knn_score": float(knn_score),
            "confidence_threshold": self.confidence_threshold,
            "knn_threshold": self.knn_threshold,
            "top3": [
                {
                    "class": self.classes[index],
                    "confidence": float(probabilities[index].item()),
                }
                for index in top_indices.tolist()
            ],
            "reason_codes": [] if accepted else [
                reason for condition, reason in [
                    (confidence.item() < self.confidence_threshold, "LOW_CONFIDENCE"),
                    (knn_score < self.knn_threshold, "LOW_SUGARCANE_SIMILARITY"),
                ] if condition
            ],
        }
'''

# Fix the top-k expression in the source string before writing.
inference_source = inference_source.replace(
    "top_indices = probabilities.topk(min(3, len(self.classes)).indices",
    "top_indices = probabilities.topk(min(3, len(self.classes))).indices",
)

(PACKAGE_DIR / "model.py").write_text(model_source.strip() + "\n")
(PACKAGE_DIR / "inference.py").write_text(inference_source.strip() + "\n")

# Save a compact inference checkpoint without optimizer state.
checkpoint = torch.load(SOURCE_CHECKPOINT, map_location="cpu", weights_only=False)
compact_checkpoint = {
    "epoch": checkpoint["epoch"],
    "model_state_dict": checkpoint["model_state_dict"],
    "class_names": checkpoint.get("class_names", CLASS_NAMES),
    "metrics": checkpoint.get("metrics", {}),
}
torch.save(compact_checkpoint, PACKAGE_DIR / "sugarcane_oe_best.pth")

# Package the training-only kNN bank, retaining no source images.
bank = torch.load(SOURCE_REFERENCE, map_location="cpu", weights_only=False)
torch.save(
    {
        "embeddings": bank["embeddings"].cpu(),
        "image_paths": bank.get("image_paths", []),
        "count": int(bank["embeddings"].shape[0]),
        "k": 10,
    },
    PACKAGE_DIR / "sugarcane_knn_reference_bank.pt",
)

shutil.copy2(
    SOURCE_RELIABILITY,
    PACKAGE_DIR / "sugarcane_reliability_config.json",
)

classes = ["Healthy", "Mosaic", "RedRot", "Rust", "Yellow"]
(PACKAGE_DIR / "classes.json").write_text(json.dumps({
    "crop": "sugarcane",
    "classes": classes,
    "class_to_idx": {name: index for index, name in enumerate(classes)},
}, indent=2))

(PACKAGE_DIR / "config.json").write_text(json.dumps({
    "crop": "sugarcane",
    "model_version": "1.0.0",
    "architecture": "SugarcaneResNet50Attention",
    "checkpoint": "sugarcane_oe_best.pth",
    "reference_bank": "sugarcane_knn_reference_bank.pt",
    "reliability_config": "sugarcane_reliability_config.json",
    "full_image_size": 448,
    "tile_size": 288,
    "tile_count": 4,
    "normalization_mean": [0.485, 0.456, 0.406],
    "normalization_std": [0.229, 0.224, 0.225],
}, indent=2))

(PACKAGE_DIR / "requirements.txt").write_text(
    "torch>=2.2\n"
    "torchvision>=0.17\n"
    "Pillow>=10.0\n"
)

(PACKAGE_DIR / "README.md").write_text(
    "# Sugarcane Disease Classifier\n\n"
    "ResNet50 with multi-scale feature fusion, spatial self-attention, "
    "and four-tile feature aggregation. The inference wrapper applies a "
    "confidence and nearest-neighbor rejection rule.\n\n"
    "Supported classes: Healthy, Mosaic, RedRot, Rust, Yellow.\n\n"
    "The rejection system was evaluated on PlantDoc other-crop images. "
    "It does not identify unsupported diseases, and results do not establish "
    "rejection performance for every unseen crop or Sugarcane condition.\n"
)

# Verify files, then load package inference code and run one training-image smoke test.
sys.path.insert(0, str(PACKAGE_DIR))
from inference import SugarcaneDiseasePredictor

predictor = SugarcaneDiseasePredictor(PACKAGE_DIR, device=str(DEVICE))
sample_path = str(pd.read_csv(EXPERIMENT_DIR / "train_manifest.csv").iloc[0]["image_path"])
sample_result = predictor.predict(sample_path)

print("Package folder:", PACKAGE_DIR)
print("Package files:")
for path in sorted(PACKAGE_DIR.iterdir()):
    print(f"- {path.name} ({path.stat().st_size / (1024 * 1024):.2f} MB)")
print("\nTraining-image smoke test:")
print("Image:", sample_path)
print("Result:", sample_result)

Package folder: /content/drive/MyDrive/final_year_project/sugarcane/sugarcane_ood_data/hf_package/sugarcane/v1
Package files:
- README.md (0.00 MB)
- __pycache__ (0.00 MB)
- classes.json (0.00 MB)
- config.json (0.00 MB)
- inference.py (0.00 MB)
- model.py (0.00 MB)
- requirements.txt (0.00 MB)
- sugarcane_knn_reference_bank.pt (3.19 MB)
- sugarcane_oe_best.pth (103.27 MB)
- sugarcane_reliability_config.json (0.00 MB)

Training-image smoke test:
Image: /content/sugarcane_raw/Sugarcane Leaf Disease Dataset/RedRot/redrot (393).jpeg
Result: {'crop': 'sugarcane', 'model_version': '1.0.0', 'decision': 'PREDICTION', 'predicted_class': 'RedRot', 'confidence': 0.9874617457389832, 'knn_score': 0.946159303188324, 'confidence_threshold': 0.57815, 'knn_threshold': 0.8175, 'top3': [{'class': 'RedRot', 'confidence': 0.9874617457389832}, {'class': 'Rust', 'confidence': 0.005380235146731138}, {'class': 'Yellow', 'confidence': 0.0029908015858381987}], 'reason_codes': []}


In [37]:
from pathlib import Path
from huggingface_hub import HfApi, notebook_login

PACKAGE_DIR = Path(
    "/content/drive/MyDrive/final_year_project/sugarcane/"
    "sugarcane_ood_data/hf_package/sugarcane/v1"
)
REPO_ID = "virajwirle/plant-disease-models"

if not PACKAGE_DIR.exists():
    raise FileNotFoundError(f"Package folder not found: {PACKAGE_DIR}")

required_files = [
    "README.md",
    "classes.json",
    "config.json",
    "inference.py",
    "model.py",
    "requirements.txt",
    "sugarcane_oe_best.pth",
    "sugarcane_knn_reference_bank.pt",
    "sugarcane_reliability_config.json",
]
missing = [name for name in required_files if not (PACKAGE_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Package is missing files: {missing}")

notebook_login()

api = HfApi()
api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    exist_ok=True,
)

upload_result = api.upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=str(PACKAGE_DIR),
    path_in_repo="sugarcane/v1",
    commit_message="Add Sugarcane disease model with OOD rejection",
)

print("Upload complete.")
print("Repository:", f"https://huggingface.co/{REPO_ID}")
print("Sugarcane model folder:", f"https://huggingface.co/{REPO_ID}/tree/main/sugarcane/v1")
print("Commit:", upload_result.commit_url)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/hf_api.py:11779: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


Upload complete.
Repository: https://huggingface.co/virajwirle/plant-disease-models
Sugarcane model folder: https://huggingface.co/virajwirle/plant-disease-models/tree/main/sugarcane/v1
Commit: https://huggingface.co/virajwirle/plant-disease-models/commit/c1982ea7bec711a4385c53812ad6e6496b6de893
